In [ ]:
# Extracts the scene context (child, action, objects, distances) of each video into a JSON file.

# Module 3 — Pipeline final d’extraction du contexte

Ce notebook contient uniquement le pipeline reproductible retenu pour le
prototype :

1. sélection déterministe de 12 vidéos RHAD et extraction de 16 frames ;
2. détection de l’enfant avec le checkpoint YOLO26m entraîné ;
3. prédiction de l’activité avec le checkpoint VideoMAE original (`Training`) ;
4. détection contextuelle open-vocabulary avec YOLOE-26s ;
5. fusion enfant–objets et calcul des distances normalisées ;
6. création directe d’un JSON définitif par vidéo ;
7. tests unitaires, d’intégration et système.

YOLOE-26s a été retenu à l’issue du benchmark séparé de 96 images. Son seuil
de confiance de 0,14 a été déterminé uniquement sur les 48 images de
calibration. Les 48 images de test ont ensuite servi à mesurer les
performances finales, sans modifier ce seuil.

## Environnement Kaggle

- Internet activé ;
- accélérateur `GPU T4 x2` ;
- datasets RHAD/Training et ChildDetection ajoutés ;
- checkpoint YOLO26m ajouté ;
- checkpoint VideoMAE original
  `videomae-final-rhad-results/checkpoint-best.pth` ajouté ;
- ne pas ajouter le checkpoint issu de `Training2` à ce pipeline final.

Le benchmark des détecteurs reste dans un notebook expérimental séparé.
Exécuter les cellules dans l’ordre avec **Run All**.


## 1 — Installation et configuration

In [ ]:
# ==============================================================================
# INSTALLATION DES DÉPENDANCES DU KERNEL KAGGLE
# ==============================================================================

%pip install --no-cache-dir -q -U ultralytics

# ==============================================================================
# VÉRIFICATION D'ULTRALYTICS
# ==============================================================================

import ultralytics
print("Ultralytics :", ultralytics.__version__)

# ==============================================================================
# CONFIGURATION GÉNÉRALE DU PIPELINE
# ==============================================================================

# ============================================================
# CELLULE 1 — VÉRIFICATION DE L'ENVIRONNEMENT ET CONFIGURATION
# AUCUNE INSTALLATION RÉSEAU AUTOMATIQUE
# ============================================================

import sys
from pathlib import Path

try:
    import ultralytics
except ModuleNotFoundError as error:
    wheel_candidates = sorted(
        Path("/kaggle/input").rglob("ultralytics*.whl")
    )

    if wheel_candidates:
        import subprocess

        print("Ultralytics absent de l'image Kaggle.")
        print("Installation depuis le wheel local :", wheel_candidates[0])

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-index",
                str(wheel_candidates[0]),
            ],
            check=True,
        )

        import ultralytics
    else:
        raise RuntimeError(
            "Ultralytics n'est pas installé et aucun wheel local n'a été "
            "trouvé dans /kaggle/input. Active Internet dans les paramètres "
            "Kaggle, redémarre la session, exécute une seule fois "
            "!pip install ultralytics, puis relance ce notebook."
        ) from error

print("Ultralytics disponible :", ultralytics.__version__)

import cv2
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO, YOLOE

# ------------------------------------------------------------
# Configuration générale définitive
# ------------------------------------------------------------

NUM_CONTEXT_FRAMES = 16
# Seuil sélectionné sur les 48 images de calibration du benchmark.
OBJECT_CONFIDENCE_THRESHOLD = 0.14
NORMAL_CLASS_INDEX = 7

# Aucun seuil de confiance n'est appliqué à l'activité. La prédiction,
# sa confiance et l'indicateur normal/anormal sont transmis au module LLM.

PROJECT_DIR = Path("/kaggle/working/P3_context_extraction")
FRAMES_DIR = PROJECT_DIR / "extracted_frames"
VISUALIZATIONS_DIR = PROJECT_DIR / "visualizations"
JSON_OUTPUTS_DIR = PROJECT_DIR / "json_outputs"
ANNOTATIONS_DIR = PROJECT_DIR / "annotations"

for directory in [
    PROJECT_DIR,
    FRAMES_DIR,
    VISUALIZATIONS_DIR,
    JSON_OUTPUTS_DIR,
    ANNOTATIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print("CONFIGURATION DU MODULE 3")
print("=" * 70)
print("Nombre de frames par vidéo :", NUM_CONTEXT_FRAMES)
print("Seuil de détection objets :", OBJECT_CONFIDENCE_THRESHOLD)
print("Proximité : distance continue, sans seuil near/far")
print("Activité VideoMAE : sortie factuelle sans seuil de décision")
print("Dossier principal :", PROJECT_DIR)
print("\n CELLULE 1 TERMINÉE")


## 2 — Sélection et extraction des frames

In [ ]:
# ==============================================================================
# INVENTAIRE DES VIDÉOS RHAD
# ==============================================================================

# ============================================================
# CELLULE 2 — RECHERCHE DES VIDÉOS RHAD DANS /kaggle/input
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# Extensions vidéo acceptées
# ------------------------------------------------------------

VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm",
    ".mpeg",
    ".mpg",
}


# ------------------------------------------------------------
# Rechercher toutes les vidéos
# ------------------------------------------------------------

KAGGLE_INPUT_DIR = Path("/kaggle/input")

video_paths = []

for path in KAGGLE_INPUT_DIR.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() in VIDEO_EXTENSIONS:
        video_paths.append(path)


print("=" * 80)
print("RECHERCHE DES VIDÉOS")
print("=" * 80)

print(
    "Nombre total de fichiers vidéo trouvés :",
    len(video_paths),
)


if len(video_paths) == 0:

    print(
        "\nAucune vidéo trouvée dans /kaggle/input."
    )

    print(
        "Vérifie que le dataset RHAD a bien été ajouté "
        "au notebook Kaggle."
    )

else:

    # --------------------------------------------------------
    # Construire un tableau récapitulatif
    # --------------------------------------------------------

    video_records = []

    for video_path in video_paths:

        relative_path = video_path.relative_to(
            KAGGLE_INPUT_DIR
        )

        path_parts = relative_path.parts

        dataset_name = (
            path_parts[0]
            if len(path_parts) >= 1
            else "unknown"
        )

        class_name = (
            video_path.parent.name
        )

        video_records.append(
            {
                "dataset_name": dataset_name,
                "class_name": class_name,
                "filename": video_path.name,
                "video_path": str(video_path),
                "extension": video_path.suffix.lower(),
            }
        )

    videos_df = pd.DataFrame(
        video_records
    )

    videos_df = videos_df.sort_values(
        by=[
            "dataset_name",
            "class_name",
            "filename",
        ]
    ).reset_index(drop=True)


    # --------------------------------------------------------
    # Afficher les datasets trouvés
    # --------------------------------------------------------

    print("\nDatasets contenant des vidéos :")

    dataset_counts = (
        videos_df
        .groupby("dataset_name")
        .size()
        .sort_values(ascending=False)
    )

    print(dataset_counts.to_string())


    # --------------------------------------------------------
    # Afficher les dossiers/classes trouvés
    # --------------------------------------------------------

    print("\nDossiers de classes trouvés :")

    class_counts = (
        videos_df
        .groupby("class_name")
        .size()
        .sort_values(ascending=False)
    )

    print(class_counts.to_string())


    # --------------------------------------------------------
    # Afficher quelques chemins
    # --------------------------------------------------------

    print("\nExemples de vidéos :")

    for video_path in videos_df[
        "video_path"
    ].head(20):

        print(video_path)


    # --------------------------------------------------------
    # Sauvegarder l’inventaire
    # --------------------------------------------------------

    VIDEO_INVENTORY_PATH = (
        PROJECT_DIR
        / "rhad_video_inventory.csv"
    )

    videos_df.to_csv(
        VIDEO_INVENTORY_PATH,
        index=False,
    )

    print(
        "\nInventaire sauvegardé dans :",
        VIDEO_INVENTORY_PATH,
    )


print("\nCELLULE 2 TERMINÉE")

# ==============================================================================
# SÉLECTION REPRODUCTIBLE DES CANDIDATS
# ==============================================================================

# ============================================================
# CELLULE 3 — PROPOSER DES VIDÉOS CANDIDATES
# ============================================================

import cv2
import random
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ------------------------------------------------------------
# Classes retenues pour le premier prototype
# ------------------------------------------------------------

TARGET_CLASSES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

CANDIDATES_PER_CLASS = 3
RANDOM_SEED = 42

random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# Fonction de lecture des métadonnées et de l’image centrale
# ------------------------------------------------------------

def read_video_information(video_path):
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        return None

    total_frames = int(
        capture.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        capture.get(cv2.CAP_PROP_FPS)
    )

    width = int(
        capture.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    duration_seconds = (
        total_frames / fps
        if fps > 0
        else None
    )

    middle_frame_index = max(
        total_frames // 2,
        0,
    )

    capture.set(
        cv2.CAP_PROP_POS_FRAMES,
        middle_frame_index,
    )

    success, frame = capture.read()

    capture.release()

    if not success:
        return None

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB,
    )

    return {
        "total_frames": total_frames,
        "fps": fps,
        "width": width,
        "height": height,
        "duration_seconds": duration_seconds,
        "middle_frame_index": middle_frame_index,
        "middle_frame_rgb": frame_rgb,
    }


# ------------------------------------------------------------
# Sélectionner trois candidats par classe
# ------------------------------------------------------------

candidate_records = []

for class_name in TARGET_CLASSES:

    class_videos = (
        videos_df[
            videos_df["class_name"] == class_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(class_videos) == 0:
        print(
            f"Aucune vidéo trouvée pour : {class_name}"
        )
        continue

    number_to_select = min(
        CANDIDATES_PER_CLASS,
        len(class_videos),
    )

    selected_indices = random.sample(
        range(len(class_videos)),
        number_to_select,
    )

    for candidate_number, dataframe_index in enumerate(
        selected_indices,
        start=1,
    ):

        video_path = Path(
            class_videos.loc[
                dataframe_index,
                "video_path",
            ]
        )

        information = read_video_information(
            video_path
        )

        if information is None:
            print(
                "Vidéo impossible à lire :",
                video_path,
            )
            continue

        candidate_records.append(
            {
                "class_name": class_name,
                "candidate_number": candidate_number,
                "filename": video_path.name,
                "video_path": str(video_path),
                "total_frames": information["total_frames"],
                "fps": information["fps"],
                "duration_seconds": information["duration_seconds"],
                "width": information["width"],
                "height": information["height"],
                "middle_frame_index": information[
                    "middle_frame_index"
                ],
                "middle_frame_rgb": information[
                    "middle_frame_rgb"
                ],
            }
        )


candidates_df = pd.DataFrame(
    candidate_records
)


# ------------------------------------------------------------
# Afficher les images centrales
# ------------------------------------------------------------

number_of_rows = len(TARGET_CLASSES)
number_of_columns = CANDIDATES_PER_CLASS

figure, axes = plt.subplots(
    number_of_rows,
    number_of_columns,
    figsize=(15, 4 * number_of_rows),
)

for row_index, class_name in enumerate(
    TARGET_CLASSES
):

    class_candidates = (
        candidates_df[
            candidates_df["class_name"] == class_name
        ]
        .sort_values("candidate_number")
        .reset_index(drop=True)
    )

    for column_index in range(
        number_of_columns
    ):

        axis = axes[
            row_index,
            column_index,
        ]

        axis.axis("off")

        if column_index >= len(
            class_candidates
        ):
            continue

        candidate = class_candidates.iloc[
            column_index
        ]

        axis.imshow(
            candidate["middle_frame_rgb"]
        )

        duration = candidate[
            "duration_seconds"
        ]

        axis.set_title(
            f"{class_name}\n"
            f"Candidat {candidate['candidate_number']}\n"
            f"{candidate['filename']}\n"
            f"Durée : {duration:.2f} s",
            fontsize=10,
        )

plt.tight_layout()

CANDIDATES_FIGURE_PATH = (
    VISUALIZATIONS_DIR
    / "video_candidates_middle_frames.png"
)

plt.savefig(
    CANDIDATES_FIGURE_PATH,
    dpi=180,
    bbox_inches="tight",
)

plt.close()


# ------------------------------------------------------------
# Sauvegarder la liste des candidats
# ------------------------------------------------------------

CANDIDATES_CSV_PATH = (
    PROJECT_DIR
    / "video_candidates.csv"
)

candidates_df.drop(
    columns=["middle_frame_rgb"]
).to_csv(
    CANDIDATES_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# Afficher le tableau récapitulatif
# ------------------------------------------------------------

display_columns = [
    "class_name",
    "candidate_number",
    "filename",
    "duration_seconds",
    "total_frames",
    "fps",
    "width",
    "height",
]

display(
    candidates_df[
        display_columns
    ]
)


print("=" * 80)
print("VIDÉOS CANDIDATES PRÉPARÉES")
print("=" * 80)

print(
    "Figure :",
    CANDIDATES_FIGURE_PATH,
)

print(
    "Tableau :",
    CANDIDATES_CSV_PATH,
)

print("\nCELLULE 3 TERMINÉE")

# ==============================================================================
# SÉLECTION DES 12 VIDÉOS ET EXTRACTION DES 192 FRAMES
# ==============================================================================

# ============================================================
# CELLULE 4 — SÉLECTION DES 12 VIDÉOS
# ET EXTRACTION DE 16 FRAMES PAR VIDÉO
# ============================================================

import cv2
import re
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


# ------------------------------------------------------------
# 1. UNE VIDÉO SÉLECTIONNÉE POUR CHAQUE CLASSE
# ------------------------------------------------------------

SELECTED_VIDEO_CHOICES = {
    "Jumping": 1,
    "child_running": 3,
    "climbing": 2,
    "collesion": 3,
    "electrical_appliances": 2,
    "falling": 2,
    "head_stuck": 2,
    "normal": 3,
    "objects_falling": 2,
    "pulling_curtains": 2,
    "scribbling": 1,
    "sharp_objects": 3,
}

EXPECTED_CLASSES = list(
    SELECTED_VIDEO_CHOICES.keys()
)

EXPECTED_NUMBER_OF_VIDEOS = 12

EXPECTED_NUMBER_OF_FRAMES = (
    EXPECTED_NUMBER_OF_VIDEOS
    * NUM_CONTEXT_FRAMES
)


# ------------------------------------------------------------
# 2. CONSTRUIRE LE TABLEAU DES VIDÉOS SÉLECTIONNÉES
# ------------------------------------------------------------

selected_rows = []

for class_name, candidate_number in (
    SELECTED_VIDEO_CHOICES.items()
):

    matching_rows = candidates_df[
        (
            candidates_df["class_name"]
            == class_name
        )
        &
        (
            candidates_df["candidate_number"]
            == candidate_number
        )
    ]

    if len(matching_rows) == 0:
        raise ValueError(
            f"Aucune vidéo trouvée pour "
            f"{class_name}, candidat {candidate_number}."
        )

    if len(matching_rows) > 1:
        raise ValueError(
            f"Plusieurs vidéos trouvées pour "
            f"{class_name}, candidat {candidate_number}."
        )

    selected_rows.append(
        matching_rows.iloc[0].copy()
    )


selected_videos_df = pd.DataFrame(
    selected_rows
).reset_index(drop=True)


# ------------------------------------------------------------
# 3. VÉRIFICATIONS DE LA SÉLECTION
# ------------------------------------------------------------

assert len(selected_videos_df) == (
    EXPECTED_NUMBER_OF_VIDEOS
), (
    f"Il faut {EXPECTED_NUMBER_OF_VIDEOS} vidéos, "
    f"mais {len(selected_videos_df)} ont été sélectionnées."
)

assert (
    selected_videos_df["class_name"].nunique()
    == EXPECTED_NUMBER_OF_VIDEOS
), (
    "Chaque classe doit apparaître exactement une fois."
)

missing_classes = set(
    EXPECTED_CLASSES
) - set(
    selected_videos_df["class_name"]
)

assert len(missing_classes) == 0, (
    f"Classes manquantes : {missing_classes}"
)


print("=" * 80)
print("VIDÉOS SÉLECTIONNÉES")
print("=" * 80)

display(
    selected_videos_df[
        [
            "class_name",
            "candidate_number",
            "filename",
            "duration_seconds",
            "total_frames",
            "fps",
            "width",
            "height",
        ]
    ]
)


# ------------------------------------------------------------
# 4. NETTOYER LES NOMS DE FICHIERS
# ------------------------------------------------------------

def sanitize_name(text):

    text = str(text)

    text = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "_",
        text,
    )

    return text.strip("_")


# ------------------------------------------------------------
# 5. EXTRAIRE 16 FRAMES UNIFORMÉMENT
# ------------------------------------------------------------

def extract_and_save_uniform_frames(
    video_path,
    class_name,
    candidate_number,
    number_of_frames=16,
):

    video_path = Path(video_path)

    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        raise RuntimeError(
            f"Impossible d’ouvrir la vidéo : "
            f"{video_path}"
        )

    total_frames = int(
        capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        capture.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        capture.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        capture.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    if total_frames <= 0:
        capture.release()

        raise RuntimeError(
            f"Nombre de frames invalide pour : "
            f"{video_path}"
        )

    if fps <= 0:
        fps = 30.0

    selected_indices = np.linspace(
        0,
        total_frames - 1,
        number_of_frames,
        dtype=int,
    )

    video_identifier = (
        f"{sanitize_name(class_name)}"
        f"__candidate_{candidate_number}"
        f"__{sanitize_name(video_path.stem)}"
    )

    video_output_dir = (
        FRAMES_DIR
        / video_identifier
    )

    # Supprimer les anciennes frames de cette vidéo
    # pour éviter de mélanger plusieurs exécutions.
    if video_output_dir.exists():
        shutil.rmtree(
            video_output_dir
        )

    video_output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    extracted_records = []

    for sample_index, frame_index in enumerate(
        selected_indices
    ):

        capture.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(frame_index),
        )

        success, frame_bgr = capture.read()

        if not success:
            print(
                "Frame impossible à lire :",
                class_name,
                frame_index,
            )

            continue

        timestamp_seconds = (
            float(frame_index)
            / float(fps)
        )

        frame_filename = (
            f"sample_{sample_index:02d}"
            f"_frame_{frame_index:05d}.jpg"
        )

        frame_path = (
            video_output_dir
            / frame_filename
        )

        save_success = cv2.imwrite(
            str(frame_path),
            frame_bgr,
        )

        if not save_success:
            raise RuntimeError(
                f"Impossible de sauvegarder : "
                f"{frame_path}"
            )

        extracted_records.append(
            {
                "video_id": video_identifier,
                "class_name": class_name,
                "candidate_number": int(
                    candidate_number
                ),
                "video_filename": video_path.name,
                "video_path": str(video_path),
                "sample_index": int(
                    sample_index
                ),
                "frame_index": int(
                    frame_index
                ),
                "timestamp_seconds": float(
                    timestamp_seconds
                ),
                "fps": float(fps),
                "width": int(width),
                "height": int(height),
                "frame_path": str(frame_path),
            }
        )

    capture.release()

    return extracted_records


# ------------------------------------------------------------
# 6. EXTRAIRE LES FRAMES DES 12 VIDÉOS
# ------------------------------------------------------------

frame_extraction_start = time.perf_counter()
all_extracted_records = []

for _, video_row in (
    selected_videos_df.iterrows()
):

    records = (
        extract_and_save_uniform_frames(
            video_path=video_row[
                "video_path"
            ],
            class_name=video_row[
                "class_name"
            ],
            candidate_number=video_row[
                "candidate_number"
            ],
            number_of_frames=(
                NUM_CONTEXT_FRAMES
            ),
        )
    )

    all_extracted_records.extend(
        records
    )


FRAME_EXTRACTION_TOTAL_SECONDS = (
    time.perf_counter() - frame_extraction_start
)

extracted_frames_df = pd.DataFrame(
    all_extracted_records
)


# ------------------------------------------------------------
# 7. COMPTER LES FRAMES PAR VIDÉO
# ------------------------------------------------------------

frames_per_video = (
    extracted_frames_df
    .groupby(
        [
            "video_id",
            "class_name",
        ]
    )
    .size()
    .reset_index(
        name="frames_extracted"
    )
)

print("\n" + "=" * 80)
print("NOMBRE DE FRAMES EXTRAITES PAR VIDÉO")
print("=" * 80)

display(
    frames_per_video
)


# ------------------------------------------------------------
# 8. VÉRIFICATIONS FINALES
# ------------------------------------------------------------

assert len(frames_per_video) == (
    EXPECTED_NUMBER_OF_VIDEOS
), (
    "Le nombre de vidéos extraites "
    "n’est pas égal à 12."
)

incorrect_frame_counts = (
    frames_per_video[
        frames_per_video[
            "frames_extracted"
        ] != NUM_CONTEXT_FRAMES
    ]
)

assert len(incorrect_frame_counts) == 0, (
    "Certaines vidéos ne contiennent pas "
    "exactement 16 frames extraites :\n"
    f"{incorrect_frame_counts}"
)

assert len(extracted_frames_df) == (
    EXPECTED_NUMBER_OF_FRAMES
), (
    f"Il fallait extraire "
    f"{EXPECTED_NUMBER_OF_FRAMES} frames, "
    f"mais {len(extracted_frames_df)} "
    f"ont été obtenues."
)


print(
    "Nombre de vidéos sélectionnées :",
    len(selected_videos_df),
)

print(
    "Nombre de frames par vidéo :",
    NUM_CONTEXT_FRAMES,
)

print(
    "Nombre total de frames :",
    len(extracted_frames_df),
)


# ------------------------------------------------------------
# 9. AFFICHER UNE GRILLE DE 16 FRAMES PAR VIDÉO
# ------------------------------------------------------------

for video_id in (
    extracted_frames_df[
        "video_id"
    ].unique()
):

    video_frames = (
        extracted_frames_df[
            extracted_frames_df[
                "video_id"
            ] == video_id
        ]
        .sort_values(
            "sample_index"
        )
        .reset_index(drop=True)
    )

    class_name = video_frames.loc[
        0,
        "class_name",
    ]

    candidate_number = video_frames.loc[
        0,
        "candidate_number",
    ]

    figure, axes = plt.subplots(
        4,
        4,
        figsize=(14, 11),
    )

    axes = axes.flatten()

    for frame_position, axis in enumerate(
        axes
    ):

        axis.axis("off")

        if frame_position >= len(
            video_frames
        ):
            continue

        frame_record = video_frames.iloc[
            frame_position
        ]

        frame_bgr = cv2.imread(
            frame_record["frame_path"]
        )

        if frame_bgr is None:
            axis.set_title(
                "Image non lisible"
            )
            continue

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        axis.imshow(
            frame_rgb
        )

        axis.set_title(
            f"Échantillon "
            f"{frame_record['sample_index']}\n"
            f"Frame "
            f"{frame_record['frame_index']} — "
            f"{frame_record['timestamp_seconds']:.2f} s",
            fontsize=8,
        )

    figure.suptitle(
        f"{class_name} — candidat "
        f"{candidate_number}",
        fontsize=15,
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.96,
        ]
    )

    montage_path = (
        VISUALIZATIONS_DIR
        / f"{video_id}_16_frames.png"
    )

    plt.savefig(
        montage_path,
        dpi=170,
        bbox_inches="tight",
    )

    plt.close()

    plt.close(
        figure
    )


# ------------------------------------------------------------
# 10. SAUVEGARDER LES TABLEAUX
# ------------------------------------------------------------

SELECTED_VIDEOS_PATH = (
    PROJECT_DIR
    / "selected_prototype_videos.csv"
)

selected_videos_df.drop(
    columns=[
        "middle_frame_rgb",
    ],
    errors="ignore",
).to_csv(
    SELECTED_VIDEOS_PATH,
    index=False,
)


EXTRACTED_FRAMES_INDEX_PATH = (
    PROJECT_DIR
    / "extracted_frames_index.csv"
)

extracted_frames_df.to_csv(
    EXTRACTED_FRAMES_INDEX_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. RÉSUMÉ FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SÉLECTION ET EXTRACTION TERMINÉES")
print("=" * 80)

print(
    "Vidéos sélectionnées :",
    SELECTED_VIDEOS_PATH,
)

print(
    "Index des frames :",
    EXTRACTED_FRAMES_INDEX_PATH,
)

print(
    "Dossier des frames :",
    FRAMES_DIR,
)

print(
    "Dossier des montages :",
    VISUALIZATIONS_DIR,
)

print(
    "\nRésultat attendu : "
    "12 vidéos et 192 frames."
)

print("\nCELLULE 4 TERMINÉE")


## 3 — Détection de l'enfant

In [ ]:
# ==============================================================================
# DÉTECTION DE L'ENFANT AVEC YOLO26M
# ==============================================================================

# ============================================================
# CELLULE 5A — DÉTECTION DE L'ENFANT AVEC YOLO26M
# SUR LES 192 FRAMES
# ============================================================

from pathlib import Path
import shutil
import time
import gc

import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from ultralytics import YOLO


# ------------------------------------------------------------
# 1. Retrouver automatiquement best (2).pt
# ------------------------------------------------------------

exact_matches = list(
    Path("/kaggle/input").rglob(
        "best (2).pt"
    )
)

print("Checkpoint exact trouvé :")

for checkpoint_path in exact_matches:
    print(checkpoint_path)


if len(exact_matches) == 0:

    possible_matches = list(
        Path("/kaggle/input").rglob(
            "best*.pt"
        )
    )

    print("\nAutres fichiers best trouvés :")

    for checkpoint_path in possible_matches:
        print(checkpoint_path)

    assert len(possible_matches) == 1, (
        "Le fichier best (2).pt n'a pas été trouvé "
        "et plusieurs checkpoints best sont disponibles."
    )

    uploaded_checkpoint_path = (
        possible_matches[0]
    )

else:

    uploaded_checkpoint_path = (
        exact_matches[0]
    )


print(
    "\nCheckpoint sélectionné :",
    uploaded_checkpoint_path,
)

print(
    "Taille :",
    f"{uploaded_checkpoint_path.stat().st_size / 1024**2:.2f} Mo",
)


# ------------------------------------------------------------
# 2. Copier sous un nom simple
# ------------------------------------------------------------

CHILD_MODEL_PATH = Path(
    "/kaggle/working/"
    "yolo26m_child_best.pt"
)

shutil.copy2(
    uploaded_checkpoint_path,
    CHILD_MODEL_PATH,
)

assert CHILD_MODEL_PATH.exists()


# ------------------------------------------------------------
# 3. Charger YOLO26m
# ------------------------------------------------------------

CHILD_CONFIDENCE_THRESHOLD = 0.25
CHILD_IMAGE_SIZE = 640

child_model = YOLO(
    str(CHILD_MODEL_PATH)
)

print(
    "\nModèle chargé :",
    CHILD_MODEL_PATH,
)

print(
    "Classes du modèle :",
    child_model.names,
)


# ------------------------------------------------------------
# 4. Préparer les dossiers de sortie
# ------------------------------------------------------------

CHILD_DETECTIONS_DIR = (
    PROJECT_DIR
    / "yolo26m_child_detections"
)

CHILD_ANNOTATED_FRAMES_DIR = (
    CHILD_DETECTIONS_DIR
    / "annotated_frames"
)

CHILD_MONTAGES_DIR = (
    CHILD_DETECTIONS_DIR
    / "montages"
)

for directory in [
    CHILD_DETECTIONS_DIR,
    CHILD_ANNOTATED_FRAMES_DIR,
    CHILD_MONTAGES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 5. Vérifier les 192 frames
# ------------------------------------------------------------

ordered_frames_df = (
    extracted_frames_df
    .sort_values(
        [
            "class_name",
            "video_id",
            "sample_index",
        ]
    )
    .reset_index(drop=True)
)

assert len(ordered_frames_df) == 192, (
    f"192 frames attendues, "
    f"mais {len(ordered_frames_df)} trouvées."
)


# ------------------------------------------------------------
# 6. Détecter l'enfant image par image
# ------------------------------------------------------------

child_detection_records = []
child_frame_summary_records = []
child_primary_inference_seconds = 0.0


for frame_position, (_, frame_row) in enumerate(

    tqdm(
        ordered_frames_df.iterrows(),
        total=len(ordered_frames_df),
        desc="Détection YOLO26m",
    ),

    start=1,
):

    video_id = str(
        frame_row["video_id"]
    )

    activity_class = str(
        frame_row["class_name"]
    )

    sample_index = int(
        frame_row["sample_index"]
    )

    frame_index = int(
        frame_row["frame_index"]
    )

    frame_path = Path(
        frame_row["frame_path"]
    )

    assert frame_path.exists(), (
        f"Frame absente : {frame_path}"
    )


    # --------------------------------------------------------
    # Inférence sur une seule frame
    # --------------------------------------------------------

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_inference_start = time.perf_counter()

    results = child_model.predict(
        source=str(frame_path),
        conf=CHILD_CONFIDENCE_THRESHOLD,
        imgsz=CHILD_IMAGE_SIZE,
        device=0,
        half=True,
        batch=1,
        verbose=False,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_primary_inference_seconds += (
        time.perf_counter() - child_inference_start
    )

    result = results[0]


    # --------------------------------------------------------
    # Sauvegarder l'image annotée
    # --------------------------------------------------------

    video_output_directory = (
        CHILD_ANNOTATED_FRAMES_DIR
        / video_id
    )

    video_output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    annotated_frame_path = (
        video_output_directory
        / frame_path.name
    )

    annotated_image = result.plot()

    cv2.imwrite(
        str(annotated_frame_path),
        annotated_image,
    )


    # --------------------------------------------------------
    # Extraire toutes les bounding boxes enfant
    # --------------------------------------------------------

    current_frame_detections = []

    if (
        result.boxes is not None
        and len(result.boxes) > 0
    ):

        for detection_index, box in enumerate(
            result.boxes
        ):

            confidence = float(
                box.conf.item()
            )

            class_id = int(
                box.cls.item()
            )

            class_name_detected = str(
                result.names[class_id]
            )

            x1, y1, x2, y2 = (
                box.xyxy[0]
                .detach()
                .cpu()
                .numpy()
                .tolist()
            )

            detection_record = {
                "video_id": video_id,
                "true_activity_class": activity_class,

                "sample_index": sample_index,
                "frame_index": frame_index,

                "timestamp_seconds": float(
                    frame_row[
                        "timestamp_seconds"
                    ]
                ),

                "frame_path": str(
                    frame_path
                ),

                "annotated_frame_path": str(
                    annotated_frame_path
                ),

                "detection_index": int(
                    detection_index
                ),

                "detected_class_id": class_id,
                "detected_class_name": (
                    class_name_detected
                ),

                "confidence": confidence,

                "bbox_x1": float(x1),
                "bbox_y1": float(y1),
                "bbox_x2": float(x2),
                "bbox_y2": float(y2),

                "bbox_width": float(
                    x2 - x1
                ),

                "bbox_height": float(
                    y2 - y1
                ),

                "detector_model": (
                    "YOLO26m child detector"
                ),
            }

            current_frame_detections.append(
                detection_record
            )

            child_detection_records.append(
                detection_record
            )


    # --------------------------------------------------------
    # Garder la détection ayant la meilleure confiance
    # --------------------------------------------------------

    if len(current_frame_detections) > 0:

        best_child_detection = max(
            current_frame_detections,
            key=lambda item: item[
                "confidence"
            ],
        )

        child_detected = True

        best_confidence = (
            best_child_detection[
                "confidence"
            ]
        )

        best_bbox_x1 = (
            best_child_detection[
                "bbox_x1"
            ]
        )

        best_bbox_y1 = (
            best_child_detection[
                "bbox_y1"
            ]
        )

        best_bbox_x2 = (
            best_child_detection[
                "bbox_x2"
            ]
        )

        best_bbox_y2 = (
            best_child_detection[
                "bbox_y2"
            ]
        )

    else:

        child_detected = False

        best_confidence = None

        best_bbox_x1 = None
        best_bbox_y1 = None
        best_bbox_x2 = None
        best_bbox_y2 = None


    child_frame_summary_records.append(
        {
            "video_id": video_id,
            "true_activity_class": activity_class,

            "sample_index": sample_index,
            "frame_index": frame_index,

            "timestamp_seconds": float(
                frame_row[
                    "timestamp_seconds"
                ]
            ),

            "frame_path": str(
                frame_path
            ),

            "annotated_frame_path": str(
                annotated_frame_path
            ),

            "child_detected": bool(
                child_detected
            ),

            "number_of_child_detections": int(
                len(
                    current_frame_detections
                )
            ),

            "child_confidence": (
                best_confidence
            ),

            "child_bbox_x1": (
                best_bbox_x1
            ),

            "child_bbox_y1": (
                best_bbox_y1
            ),

            "child_bbox_x2": (
                best_bbox_x2
            ),

            "child_bbox_y2": (
                best_bbox_y2
            ),
        }
    )


    del results
    del result
    del annotated_image


    if frame_position % 16 == 0:

        gc.collect()

        torch.cuda.empty_cache()

        print(
            f"{frame_position} / 192 "
            "frames terminées"
        )


# ------------------------------------------------------------
# 7. Créer les tableaux
# ------------------------------------------------------------

child_detections_df = pd.DataFrame(
    child_detection_records
)

child_frame_summary_df = pd.DataFrame(
    child_frame_summary_records
)

assert len(
    child_frame_summary_df
) == 192


# ------------------------------------------------------------
# 8. Résumé par vidéo
# ------------------------------------------------------------

child_video_summary_df = (
    child_frame_summary_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ]
    )
    .agg(
        frames_analyzed=(
            "frame_path",
            "size",
        ),

        frames_with_child=(
            "child_detected",
            "sum",
        ),

        total_child_detections=(
            "number_of_child_detections",
            "sum",
        ),

        mean_child_confidence=(
            "child_confidence",
            "mean",
        ),

        maximum_child_confidence=(
            "child_confidence",
            "max",
        ),
    )
    .reset_index()
)

child_video_summary_df[
    "child_detection_rate"
] = (
    child_video_summary_df[
        "frames_with_child"
    ]
    /
    child_video_summary_df[
        "frames_analyzed"
    ]
)


# ------------------------------------------------------------
# 9. Sauvegarder les CSV
# ------------------------------------------------------------

CHILD_DETECTIONS_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_detections.csv"
)

CHILD_FRAME_SUMMARY_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_frame_summary.csv"
)

CHILD_VIDEO_SUMMARY_CSV_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_video_summary.csv"
)

child_detections_df.to_csv(
    CHILD_DETECTIONS_CSV_PATH,
    index=False,
)

child_frame_summary_df.to_csv(
    CHILD_FRAME_SUMMARY_CSV_PATH,
    index=False,
)

child_video_summary_df.to_csv(
    CHILD_VIDEO_SUMMARY_CSV_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Créer quatre images par vidéo sans les afficher
# ------------------------------------------------------------

MONTAGE_SAMPLE_INDICES = [
    0,
    5,
    10,
    15,
]


for video_id in (
    child_frame_summary_df[
        "video_id"
    ].unique()
):

    video_frames = (
        child_frame_summary_df[
            child_frame_summary_df[
                "video_id"
            ] == video_id
        ]
        .sort_values(
            "sample_index"
        )
        .reset_index(drop=True)
    )

    activity_class = str(
        video_frames.loc[
            0,
            "true_activity_class",
        ]
    )

    montage_frames = (
        video_frames[
            video_frames[
                "sample_index"
            ].isin(
                MONTAGE_SAMPLE_INDICES
            )
        ]
        .sort_values(
            "sample_index"
        )
        .reset_index(drop=True)
    )

    figure, axes = plt.subplots(
        2,
        2,
        figsize=(12, 9),
    )

    axes = axes.flatten()

    for axis_index, axis in enumerate(
        axes
    ):

        axis.axis("off")

        if axis_index >= len(
            montage_frames
        ):
            continue

        frame_record = montage_frames.iloc[
            axis_index
        ]

        image_bgr = cv2.imread(
            frame_record[
                "annotated_frame_path"
            ]
        )

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        axis.imshow(
            image_rgb
        )

        confidence = frame_record[
            "child_confidence"
        ]

        confidence_text = (
            f"{confidence:.3f}"
            if pd.notna(confidence)
            else "aucune"
        )

        axis.set_title(
            f"Échantillon "
            f"{frame_record['sample_index']}\n"
            f"Enfant détecté : "
            f"{frame_record['child_detected']} — "
            f"confiance : {confidence_text}",
            fontsize=9,
        )

    figure.suptitle(
        f"{activity_class} — YOLO26m",
        fontsize=14,
    )

    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.95,
        ]
    )

    montage_path = (
        CHILD_MONTAGES_DIR
        / f"{video_id}_child_detection.png"
    )

    plt.savefig(
        montage_path,
        dpi=170,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# ------------------------------------------------------------
# 11. Afficher seulement le résumé
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("YOLO26M — DÉTECTION DE L'ENFANT TERMINÉE")
print("=" * 80)

display(
    child_video_summary_df[
        [
            "true_activity_class",
            "frames_analyzed",
            "frames_with_child",
            "child_detection_rate",
            "mean_child_confidence",
            "maximum_child_confidence",
        ]
    ]
)

print(
    "\nFrames analysées :",
    len(child_frame_summary_df),
)

print(
    "CSV par frame :",
    CHILD_FRAME_SUMMARY_CSV_PATH,
)

print(
    "CSV par vidéo :",
    CHILD_VIDEO_SUMMARY_CSV_PATH,
)

print(
    "Montages :",
    CHILD_MONTAGES_DIR,
)

print("\nCELLULE 5A TERMINÉE")

# ==============================================================================
# SECONDE PASSE SUR LES FRAMES NON DÉTECTÉES
# ==============================================================================

# ============================================================
# CELLULE 5B — SECONDE TENTATIVE SUR LES FRAMES NON DÉTECTÉES
# ============================================================

from pathlib import Path

import cv2
import pandas as pd
import torch

from tqdm.auto import tqdm
from ultralytics import YOLO


# ------------------------------------------------------------
# 1. Configuration du seuil de secours
# ------------------------------------------------------------

FALLBACK_CHILD_CONFIDENCE_THRESHOLD = 0.10
FALLBACK_CHILD_IMAGE_SIZE = 640


# ------------------------------------------------------------
# 2. Recharger le modèle seulement si nécessaire
# ------------------------------------------------------------

if "child_model" not in globals():

    assert CHILD_MODEL_PATH.exists(), (
        f"Modèle introuvable : {CHILD_MODEL_PATH}"
    )

    child_model = YOLO(
        str(CHILD_MODEL_PATH)
    )


# ------------------------------------------------------------
# 3. Copier le tableau principal
# ------------------------------------------------------------

child_frame_summary_fallback_df = (
    child_frame_summary_df
    .copy()
)


child_frame_summary_fallback_df[
    "child_detection_source"
] = "none"

child_frame_summary_fallback_df.loc[
    child_frame_summary_fallback_df[
        "child_detected"
    ] == True,
    "child_detection_source",
] = "primary_threshold_0.25"


# ------------------------------------------------------------
# 4. Identifier uniquement les frames manquées
# ------------------------------------------------------------

missed_frames_df = (
    child_frame_summary_fallback_df[
        child_frame_summary_fallback_df[
            "child_detected"
        ] == False
    ]
    .copy()
)


print("=" * 80)
print("SECONDE TENTATIVE")
print("=" * 80)

print(
    "Frames non détectées lors du premier passage :",
    len(missed_frames_df),
)


# ------------------------------------------------------------
# 5. Dossier des images de secours
# ------------------------------------------------------------

FALLBACK_ANNOTATED_DIR = (
    CHILD_DETECTIONS_DIR
    / "fallback_annotated_frames"
)

FALLBACK_ANNOTATED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 6. Relancer seulement les frames manquées
# ------------------------------------------------------------

fallback_detection_records = []
child_fallback_inference_seconds = 0.0


for dataframe_index, frame_row in tqdm(
    missed_frames_df.iterrows(),
    total=len(missed_frames_df),
    desc="Seconde tentative YOLO26m",
):

    frame_path = Path(
        frame_row["frame_path"]
    )

    assert frame_path.exists(), (
        f"Frame introuvable : {frame_path}"
    )


    # Aucun half=True :
    # cela évite les avertissements affichés précédemment.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    fallback_inference_start = time.perf_counter()

    results = child_model.predict(
        source=str(frame_path),
        conf=FALLBACK_CHILD_CONFIDENCE_THRESHOLD,
        imgsz=FALLBACK_CHILD_IMAGE_SIZE,
        device=0,
        batch=1,
        verbose=False,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    child_fallback_inference_seconds += (
        time.perf_counter() - fallback_inference_start
    )

    result = results[0]


    # --------------------------------------------------------
    # Vérifier si une détection existe
    # --------------------------------------------------------

    if (
        result.boxes is None
        or len(result.boxes) == 0
    ):
        continue


    # --------------------------------------------------------
    # Garder la détection de meilleure confiance
    # --------------------------------------------------------

    best_box = max(
        result.boxes,
        key=lambda box: float(
            box.conf.item()
        ),
    )

    confidence = float(
        best_box.conf.item()
    )

    class_id = int(
        best_box.cls.item()
    )

    class_name = str(
        result.names[class_id]
    )

    x1, y1, x2, y2 = (
        best_box.xyxy[0]
        .detach()
        .cpu()
        .numpy()
        .tolist()
    )


    # --------------------------------------------------------
    # Mettre à jour le tableau principal
    # --------------------------------------------------------

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_detected",
    ] = True

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "number_of_child_detections",
    ] = int(
        len(result.boxes)
    )

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_confidence",
    ] = confidence

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_x1",
    ] = float(x1)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_y1",
    ] = float(y1)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_x2",
    ] = float(x2)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_bbox_y2",
    ] = float(y2)

    child_frame_summary_fallback_df.loc[
        dataframe_index,
        "child_detection_source",
    ] = "fallback_threshold_0.10"


    # --------------------------------------------------------
    # Sauvegarder l’image annotée du second passage
    # --------------------------------------------------------

    video_fallback_dir = (
        FALLBACK_ANNOTATED_DIR
        / str(
            frame_row["video_id"]
        )
    )

    video_fallback_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    fallback_image_path = (
        video_fallback_dir
        / frame_path.name
    )

    annotated_image = result.plot()

    cv2.imwrite(
        str(fallback_image_path),
        annotated_image,
    )


    fallback_detection_records.append(
        {
            "video_id": str(
                frame_row["video_id"]
            ),

            "true_activity_class": str(
                frame_row[
                    "true_activity_class"
                ]
            ),

            "sample_index": int(
                frame_row["sample_index"]
            ),

            "frame_index": int(
                frame_row["frame_index"]
            ),

            "frame_path": str(
                frame_path
            ),

            "fallback_annotated_path": str(
                fallback_image_path
            ),

            "detected_class_name": class_name,

            "confidence": confidence,

            "bbox_x1": float(x1),
            "bbox_y1": float(y1),
            "bbox_x2": float(x2),
            "bbox_y2": float(y2),

            "detection_source": (
                "fallback_threshold_0.10"
            ),
        }
    )


    del results
    del result
    del annotated_image


# ------------------------------------------------------------
# 7. Créer le résumé final par vidéo
# ------------------------------------------------------------

child_video_summary_fallback_df = (
    child_frame_summary_fallback_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ]
    )
    .agg(
        frames_analyzed=(
            "frame_path",
            "size",
        ),

        frames_with_child=(
            "child_detected",
            "sum",
        ),

        mean_child_confidence=(
            "child_confidence",
            "mean",
        ),

        maximum_child_confidence=(
            "child_confidence",
            "max",
        ),

        fallback_detections=(
            "child_detection_source",
            lambda values: int(
                (
                    values
                    == "fallback_threshold_0.10"
                ).sum()
            ),
        ),
    )
    .reset_index()
)


child_video_summary_fallback_df[
    "child_detection_rate"
] = (
    child_video_summary_fallback_df[
        "frames_with_child"
    ]
    /
    child_video_summary_fallback_df[
        "frames_analyzed"
    ]
)


# ------------------------------------------------------------
# 8. Calculer le résultat global
# ------------------------------------------------------------

total_detected_before = int(
    child_frame_summary_df[
        "child_detected"
    ].sum()
)

total_detected_after = int(
    child_frame_summary_fallback_df[
        "child_detected"
    ].sum()
)

new_detections = (
    total_detected_after
    - total_detected_before
)


# ------------------------------------------------------------
# 9. Sauvegarder les nouveaux résultats
# ------------------------------------------------------------

FALLBACK_FRAME_SUMMARY_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_frame_summary_with_fallback.csv"
)

FALLBACK_VIDEO_SUMMARY_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_child_video_summary_with_fallback.csv"
)

FALLBACK_DETECTIONS_PATH = (
    CHILD_DETECTIONS_DIR
    / "yolo26m_fallback_detections.csv"
)


child_frame_summary_fallback_df.to_csv(
    FALLBACK_FRAME_SUMMARY_PATH,
    index=False,
)

child_video_summary_fallback_df.to_csv(
    FALLBACK_VIDEO_SUMMARY_PATH,
    index=False,
)

pd.DataFrame(
    fallback_detection_records
).to_csv(
    FALLBACK_DETECTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Afficher le résumé
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RÉSULTATS APRÈS LE SEUIL DE SECOURS")
print("=" * 80)

display(
    child_video_summary_fallback_df[
        [
            "true_activity_class",
            "frames_analyzed",
            "frames_with_child",
            "child_detection_rate",
            "fallback_detections",
            "mean_child_confidence",
        ]
    ]
)


print(
    "\nDétections avant :",
    f"{total_detected_before} / 192",
)

print(
    "Nouvelles détections obtenues :",
    new_detections,
)

print(
    "Détections après :",
    f"{total_detected_after} / 192",
)

print(
    "Taux global après :",
    f"{100 * total_detected_after / 192:.2f} %-",
)

print(
    "\nCSV final par frame :",
    FALLBACK_FRAME_SUMMARY_PATH,
)

print(
    "CSV final par vidéo :",
    FALLBACK_VIDEO_SUMMARY_PATH,
)

print("\nCELLULE 5B TERMINÉE")
CHILD_INFERENCE_TOTAL_SECONDS = (
    child_primary_inference_seconds
    + child_fallback_inference_seconds
)
print("Temps total d'inférence YOLO26m :", f"{CHILD_INFERENCE_TOTAL_SECONDS:.3f} s")


## 4 — Préparation de VideoMAE

In [ ]:
# ==============================================================================
# PRÉPARATION ISOLÉE DE VIDEOMAE
# ==============================================================================

# ============================================================
# MODULE 3 — ÉTAPE 2
# PRÉPARATION DU PREMIER VIDEOMAE POUR L'INFÉRENCE
# ============================================================

from pathlib import Path
import shutil
import subprocess
import sys


# ============================================================
# 1. VÉRIFIER QUE LES ÉTAPES PRÉCÉDENTES SONT TERMINÉES
# ============================================================

assert "selected_videos_df" in globals(), (
    "Exécute d'abord les cellules de sélection "
    "des 12 vidéos."
)

assert "extracted_frames_df" in globals(), (
    "Exécute d'abord les cellules d'extraction "
    "des 192 frames."
)

assert len(selected_videos_df) == 12
assert len(extracted_frames_df) == 192


# ============================================================
# 2. RETROUVER LE PREMIER CHECKPOINT VIDEOMAE
# ============================================================

checkpoint_candidates = [
    path
    for path in Path("/kaggle/input").rglob(
        "checkpoint-best.pth"
    )
    if (
        "videomae-final-rhad-results"
        in str(path).lower()
    )
]

print(
    "Checkpoints originaux trouvés :",
    len(checkpoint_candidates),
)

for path in checkpoint_candidates:
    print(" -", path)

assert len(checkpoint_candidates) == 1, (
    "Un seul checkpoint du premier VideoMAE "
    f"était attendu, trouvé : "
    f"{len(checkpoint_candidates)}"
)

VIDEOMAE_CHECKPOINT = (
    checkpoint_candidates[0]
)

assert VIDEOMAE_CHECKPOINT.is_file()

assert (
    "training2"
    not in str(VIDEOMAE_CHECKPOINT).lower()
)

print(
    "\nCheckpoint sélectionné :",
    VIDEOMAE_CHECKPOINT,
)


# ============================================================
# 3. CONFIGURATION DE L'ENVIRONNEMENT ISOLÉ
# ============================================================

VIDEOMAE_ENV_DIR = Path(
    "/kaggle/working/videomae_inference_env"
)

VIDEOMAE_PYTHON = (
    VIDEOMAE_ENV_DIR
    / "bin"
    / "python"
)

VIDEOMAE_REPO_DIR = Path(
    "/kaggle/working/VideoMAEv2"
)

VIDEOMAE_COMMIT = (
    "29eab1e8a588d1b3ec0cdec7b03a86cca491b74b"
)


# ============================================================
# 4. INSTALLER UV
# ============================================================

if shutil.which("uv") is None:

    print(
        "\nInstallation de uv..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "uv",
        ],
        check=True,
    )

UV_EXECUTABLE = shutil.which("uv")

assert UV_EXECUTABLE is not None

print(
    "uv :",
    UV_EXECUTABLE,
)


# ============================================================
# 5. CRÉER L'ENVIRONNEMENT PYTHON 3.10
# ============================================================

if not VIDEOMAE_PYTHON.exists():

    print(
        "\nCréation de l'environnement "
        "VideoMAE Python 3.10..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "python",
            "install",
            "3.10",
        ],
        check=True,
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "venv",
            "--python",
            "3.10",
            str(VIDEOMAE_ENV_DIR),
        ],
        check=True,
    )

else:

    print(
        "\nL'environnement VideoMAE existe déjà."
    )

assert VIDEOMAE_PYTHON.exists()

print(
    "Python VideoMAE :",
    VIDEOMAE_PYTHON,
)


# ============================================================
# 6. VÉRIFIER LES DÉPENDANCES
# ============================================================

dependency_test = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        (
            "import torch, torchvision, timm, "
            "numpy, decord, cv2, einops, scipy, "
            "pandas, tensorboardX, sklearn"
        ),
    ],
    text=True,
    capture_output=True,
)


# ============================================================
# 7. INSTALLER LES DÉPENDANCES SI NÉCESSAIRE
# ============================================================

if dependency_test.returncode != 0:

    print(
        "\nInstallation de PyTorch..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "pip",
            "install",
            "--python",
            str(VIDEOMAE_PYTHON),
            "--index-url",
            "https://download.pytorch.org/whl/cu118",

            "torch==2.1.2",
            "torchvision==0.16.2",
            "torchaudio==2.1.2",
        ],
        check=True,
    )

    print(
        "\nInstallation des dépendances "
        "VideoMAE..."
    )

    subprocess.run(
        [
            UV_EXECUTABLE,
            "pip",
            "install",
            "--python",
            str(VIDEOMAE_PYTHON),

            "numpy==1.26.4",
            "timm==0.4.12",
            "decord==0.6.0",
            "av",
            "opencv-python-headless",
            "Pillow",
            "einops",
            "scipy",
            "pandas",
            "tensorboardX",
            "matplotlib",
            "scikit-learn",
            "packaging",
        ],
        check=True,
    )

else:

    print(
        "\nLes dépendances VideoMAE "
        "sont déjà installées."
    )


# ============================================================
# 8. TESTER PYTORCH ET LE GPU
# ============================================================

gpu_test_code = r"""
import sys
import torch
import torchvision
import timm
import numpy
import decord

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)
print("timm        :", timm.__version__)
print("NumPy       :", numpy.__version__)
print("Decord      :", decord.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU count   :", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n'est pas disponible."
    )

for index in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {index} :",
        torch.cuda.get_device_name(index)
    )

x = torch.randn(
    64,
    64,
    device="cuda",
)

y = x @ x.T

assert torch.isfinite(y).all()

print("Calcul GPU : OK")
"""

gpu_test_result = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        gpu_test_code,
    ],
    text=True,
    capture_output=True,
)

print(
    "\n" + gpu_test_result.stdout
)

if gpu_test_result.stderr:

    print(
        "Messages système :"
    )

    print(
        gpu_test_result.stderr
    )

assert gpu_test_result.returncode == 0, (
    "Le test de l'environnement GPU "
    "VideoMAE a échoué."
)


# ============================================================
# 9. TÉLÉCHARGER LE DÉPÔT VIDEOMAEV2
# ============================================================

if not (
    VIDEOMAE_REPO_DIR
    / ".git"
).exists():

    print(
        "\nTéléchargement de VideoMAEv2..."
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/"
            "OpenGVLab/VideoMAEv2.git",
            str(VIDEOMAE_REPO_DIR),
        ],
        check=True,
    )

else:

    print(
        "\nLe dépôt VideoMAEv2 existe déjà."
    )


# ============================================================
# 10. UTILISER LA VERSION EXACTE DE L'ENTRAÎNEMENT
# ============================================================

subprocess.run(
    [
        "git",
        "checkout",
        VIDEOMAE_COMMIT,
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    check=True,
)

actual_commit = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    text=True,
).strip()

assert actual_commit == VIDEOMAE_COMMIT

print(
    "Commit VideoMAE :",
    actual_commit,
)


# ============================================================
# 11. VÉRIFIER LES FICHIERS DU DÉPÔT
# ============================================================

required_files = [
    VIDEOMAE_REPO_DIR
    / "run_class_finetuning.py",

    VIDEOMAE_REPO_DIR
    / "engine_for_finetuning.py",

    VIDEOMAE_REPO_DIR
    / "dataset"
    / "build.py",

    VIDEOMAE_REPO_DIR
    / "dataset"
    / "datasets.py",

    VIDEOMAE_REPO_DIR
    / "models"
    / "modeling_finetune.py",
]

for required_file in required_files:

    assert required_file.exists(), (
        f"Fichier absent : "
        f"{required_file}"
    )


# ============================================================
# 12. TESTER LA CRÉATION DE VIDEOMAE
# ============================================================

model_test_code = r"""
import timm
import models

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_path_rate=0.1,
    use_mean_pooling=True,
    init_scale=0.001,
)

classifier = model.get_classifier()

assert classifier.out_features == 12

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Modèle :",
    model.__class__.__name__,
)

print(
    "Paramètres :",
    number_of_parameters,
)

print(
    "Classes :",
    classifier.out_features,
)

print(
    "Création VideoMAE : OK",
)
"""

model_test_result = subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-c",
        model_test_code,
    ],
    cwd=str(VIDEOMAE_REPO_DIR),
    text=True,
    capture_output=True,
)

print(
    "\n" + model_test_result.stdout
)

if model_test_result.stderr:

    print(
        "Messages système du modèle :"
    )

    print(
        model_test_result.stderr
    )

assert model_test_result.returncode == 0, (
    "La création de VideoMAE a échoué."
)


# ============================================================
# 13. RÉSUMÉ FINAL
# ============================================================

print("\n" + "=" * 80)
print("VIDEOMAE PRÊT POUR L'INFÉRENCE")
print("=" * 80)

print(
    "Checkpoint :",
    VIDEOMAE_CHECKPOINT,
)

print(
    "Python :",
    VIDEOMAE_PYTHON,
)

print(
    "Dépôt :",
    VIDEOMAE_REPO_DIR,
)

print(
    "Commit :",
    actual_commit,
)

print(
    "\n PRÉPARATION VIDEOMAE TERMINÉE"
)

## 5 — Inférence VideoMAE

In [ ]:
# ============================================================================
# INFÉRENCE VIDEOMAE SUR LES 12 VIDÉOS SÉLECTIONNÉES
# ============================================================================

from pathlib import Path
import os
import shutil
import subprocess
import time

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------
# 1. Configuration et mapping exact des 12 classes
# ----------------------------------------------------------------------------

VIDEOMAE_CLASS_NAMES = [
    "Jumping",
    "child_running",
    "climbing",
    "collesion",
    "electrical_appliances",
    "falling",
    "head_stuck",
    "normal",
    "objects_falling",
    "pulling_curtains",
    "scribbling",
    "sharp_objects",
]

VIDEOMAE_CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(VIDEOMAE_CLASS_NAMES)
}

assert VIDEOMAE_CLASS_TO_INDEX["normal"] == NORMAL_CLASS_INDEX == 7
assert Path(VIDEOMAE_PYTHON).exists(), VIDEOMAE_PYTHON
assert VIDEOMAE_REPO_DIR.exists(), VIDEOMAE_REPO_DIR
assert VIDEOMAE_CHECKPOINT.exists(), VIDEOMAE_CHECKPOINT


# ----------------------------------------------------------------------------
# 2. Adapter le dépôt à RHAD, sans DeepSpeed obligatoire
# ----------------------------------------------------------------------------

RUN_FILE = VIDEOMAE_REPO_DIR / "run_class_finetuning.py"
BUILD_FILE = VIDEOMAE_REPO_DIR / "dataset" / "build.py"

run_text = RUN_FILE.read_text(encoding="utf-8")

if "try:\n    import deepspeed" not in run_text:
    assert "import deepspeed" in run_text
    run_text = run_text.replace(
        "import deepspeed",
        """try:
    import deepspeed
except ImportError:
    deepspeed = None""",
        1,
    )

old_deepspeed_block = """    if known_args.enable_deepspeed:
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

new_deepspeed_block = """    if known_args.enable_deepspeed:
        if deepspeed is None:
            raise ImportError("DeepSpeed demandé mais non installé.")
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

if old_deepspeed_block in run_text:
    run_text = run_text.replace(
        old_deepspeed_block,
        new_deepspeed_block,
        1,
    )

argument_start = run_text.find(
    "parser.add_argument(\n        '--data_set'"
)
assert argument_start != -1, "Argument --data_set introuvable."

argument_end = run_text.find("type=str", argument_start)
argument_block = run_text[argument_start:argument_end]

if "'RHAD'" not in argument_block:
    assert "'MIT'" in argument_block
    patched_argument_block = argument_block.replace(
        "'MIT'",
        "'MIT', 'RHAD'",
        1,
    )
    run_text = (
        run_text[:argument_start]
        + patched_argument_block
        + run_text[argument_end:]
    )

RUN_FILE.write_text(run_text, encoding="utf-8")

build_text = BUILD_FILE.read_text(encoding="utf-8")

old_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'val.csv')
"""

new_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'test.csv')
"""

if old_test_block in build_text:
    build_text = build_text.replace(
        old_test_block,
        new_test_block,
        1,
    )

rhad_branch = """
    elif args.data_set == 'RHAD':
        dataset = VideoClsDataset(
            anno_path=anno_path,
            data_root=args.data_root,
            mode=mode,
            clip_len=args.num_frames,
            frame_sample_rate=args.sampling_rate,
            num_segment=1,
            test_num_segment=args.test_num_segment,
            test_num_crop=args.test_num_crop,
            num_crop=1 if not test_mode else 3,
            keep_aspect_ratio=True,
            crop_size=args.input_size,
            short_side_size=args.short_side_size,
            new_height=256,
            new_width=320,
            sparse_sample=False,
            args=args,
        )
        nb_classes = 12
"""

unsupported_block = """    else:
        raise NotImplementedError('Unsupported Dataset')
"""

if "elif args.data_set == 'RHAD':" not in build_text:
    assert unsupported_block in build_text
    build_text = build_text.replace(
        unsupported_block,
        rhad_branch + "\n" + unsupported_block,
        1,
    )

BUILD_FILE.write_text(build_text, encoding="utf-8")

subprocess.run(
    [
        str(VIDEOMAE_PYTHON),
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(BUILD_FILE),
    ],
    check=True,
)


# ----------------------------------------------------------------------------
# 3. Préparer les 12 vidéos et les annotations d'inférence
# ----------------------------------------------------------------------------

videomae_selected_df = (
    extracted_frames_df[
        ["video_id", "class_name", "video_path"]
    ]
    .drop_duplicates(subset=["video_id"])
    .sort_values("video_id")
    .reset_index(drop=True)
)

assert len(videomae_selected_df) == 12
assert set(videomae_selected_df["class_name"]) == set(VIDEOMAE_CLASS_NAMES)

VIDEOMAE_INFERENCE_DIR = Path(
    "/kaggle/working/P3_videomae_inference"
)
VIDEOMAE_VIDEO_ROOT = VIDEOMAE_INFERENCE_DIR / "videos"
VIDEOMAE_ANNOTATIONS_DIR = VIDEOMAE_INFERENCE_DIR / "annotations"
VIDEOMAE_RAW_OUTPUT_DIR = VIDEOMAE_INFERENCE_DIR / "raw_predictions"

if VIDEOMAE_INFERENCE_DIR.exists():
    shutil.rmtree(VIDEOMAE_INFERENCE_DIR)

for directory in [
    VIDEOMAE_VIDEO_ROOT,
    VIDEOMAE_ANNOTATIONS_DIR,
    VIDEOMAE_RAW_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

annotation_lines = []
inference_records = []

for _, row in videomae_selected_df.iterrows():
    video_id = str(row["video_id"])
    true_class = str(row["class_name"])
    source_path = Path(row["video_path"])

    assert source_path.exists(), source_path
    label = VIDEOMAE_CLASS_TO_INDEX[true_class]
    alias_filename = f"{video_id}{source_path.suffix.lower()}"
    alias_path = VIDEOMAE_VIDEO_ROOT / alias_filename

    os.symlink(str(source_path), str(alias_path))
    annotation_lines.append(f"{alias_filename} {label}\n")
    inference_records.append(
        {
            "video_id": video_id,
            "alias_filename": alias_filename,
            "video_path": str(source_path),
            "true_label": int(label),
            "true_class": true_class,
        }
    )

videomae_inference_index_df = pd.DataFrame(inference_records)

for filename in ["train.csv", "val.csv", "test.csv"]:
    (VIDEOMAE_ANNOTATIONS_DIR / filename).write_text(
        "".join(annotation_lines),
        encoding="utf-8",
    )

videomae_inference_index_df.to_csv(
    VIDEOMAE_INFERENCE_DIR / "videomae_inference_index.csv",
    index=False,
)


# ----------------------------------------------------------------------------
# 4. Exécuter le checkpoint original sur deux GPU T4
# ----------------------------------------------------------------------------

command = [
    str(VIDEOMAE_PYTHON),
    "-m",
    "torch.distributed.run",
    "--nproc_per_node=2",
    "--master_port=29621",
    "run_class_finetuning.py",
    "--model", "vit_base_patch16_224",
    "--nb_classes", "12",
    "--use_mean_pooling",
    "--init_scale", "0.001",
    "--data_set", "RHAD",
    "--data_path", str(VIDEOMAE_ANNOTATIONS_DIR),
    "--data_root", str(VIDEOMAE_VIDEO_ROOT),
    "--input_size", "224",
    "--short_side_size", "224",
    "--num_frames", "16",
    "--num_segments", "1",
    "--tubelet_size", "2",
    "--sampling_rate", "4",
    "--batch_size", "2",
    "--num_workers", "2",
    "--seed", "42",
    "--test_num_segment", "2",
    "--test_num_crop", "3",
    "--dist_eval",
    "--finetune", str(VIDEOMAE_CHECKPOINT),
    "--model_key", "model",
    "--no_auto_resume",
    "--eval",
    "--output_dir", str(VIDEOMAE_RAW_OUTPUT_DIR),
    "--log_dir", str(VIDEOMAE_RAW_OUTPUT_DIR),
]

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["OMP_NUM_THREADS"] = "1"
environment["PYTHONUNBUFFERED"] = "1"

log_path = VIDEOMAE_INFERENCE_DIR / "videomae_inference_console.log"

print("=" * 80)
print("INFÉRENCE VIDEOMAE — CHECKPOINT ORIGINAL")
print("=" * 80)
print("Checkpoint :", VIDEOMAE_CHECKPOINT)
print("Vidéos     :", len(videomae_inference_index_df))

start_time = time.perf_counter()

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd=str(VIDEOMAE_REPO_DIR),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

total_inference_time = time.perf_counter() - start_time

if return_code != 0:
    raise RuntimeError(
        "L'inférence VideoMAE a échoué. "
        f"Code de retour : {return_code}"
    )


# ----------------------------------------------------------------------------
# 5. Agréger les six vues de test de chaque vidéo
# ----------------------------------------------------------------------------

raw_records = []
raw_files = [
    VIDEOMAE_RAW_OUTPUT_DIR / "0.txt",
    VIDEOMAE_RAW_OUTPUT_DIR / "1.txt",
]

assert all(path.exists() for path in raw_files), (
    "Prédictions brutes VideoMAE absentes."
)

for raw_file in raw_files:
    lines = raw_file.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines()

    for line in lines:
        if "[" not in line or "]" not in line:
            continue

        video_part, remaining = line.split("[", 1)
        logits_text, metadata_text = remaining.split("]", 1)

        logits = np.fromstring(
            logits_text,
            dtype=np.float64,
            sep=",",
        )
        metadata = metadata_text.strip().split()

        if len(logits) != 12 or len(metadata) < 3:
            continue

        record = {
            "video_id": Path(video_part.strip()).stem,
            "true_label": int(metadata[0]),
            "segment_index": int(metadata[1]),
            "crop_index": int(metadata[2]),
        }

        for class_index in range(12):
            record[f"logit_{class_index}"] = float(logits[class_index])

        raw_records.append(record)

videomae_raw_predictions_df = pd.DataFrame(raw_records)
expected_views = 12 * 2 * 3

assert len(videomae_raw_predictions_df) == expected_views, (
    f"{expected_views} vues attendues, "
    f"{len(videomae_raw_predictions_df)} trouvées."
)


def stable_softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits)
    values = np.exp(logits)
    return values / values.sum()


logit_columns = [f"logit_{index}" for index in range(12)]
prediction_records = []

for _, reference_row in videomae_inference_index_df.iterrows():
    video_id = str(reference_row["video_id"])
    video_views = videomae_raw_predictions_df[
        videomae_raw_predictions_df["video_id"] == video_id
    ]

    assert len(video_views) == 6, (
        f"{video_id} : {len(video_views)} vues au lieu de 6."
    )

    mean_logits = (
        video_views[logit_columns]
        .to_numpy(dtype=np.float64)
        .mean(axis=0)
    )
    probabilities = stable_softmax(mean_logits)
    predicted_label = int(np.argmax(probabilities))
    predicted_class = VIDEOMAE_CLASS_NAMES[predicted_label]
    confidence = float(probabilities[predicted_label])
    is_abnormal = predicted_class != "normal"

    prediction_records.append(
        {
            "video_id": video_id,
            "video_path": str(reference_row["video_path"]),
            "reference_class": str(reference_row["true_class"]),
            "predicted_label": predicted_label,
            "predicted_class": predicted_class,
            "confidence": confidence,
            "is_abnormal": bool(is_abnormal),
            "correct_prediction": bool(
                predicted_class == reference_row["true_class"]
            ),
        }
    )

videomae_predictions_df = pd.DataFrame(prediction_records)
mean_time_per_video = total_inference_time / len(videomae_predictions_df)
videomae_predictions_df[
    "mean_processing_time_per_video_seconds"
] = float(mean_time_per_video)

assert len(videomae_predictions_df) == 12
assert videomae_predictions_df["video_id"].nunique() == 12
assert videomae_predictions_df["confidence"].between(0, 1).all()

VIDEOMAE_PREDICTIONS_PATH = (
    VIDEOMAE_INFERENCE_DIR / "videomae_predictions_12_videos.csv"
)
videomae_predictions_df.to_csv(
    VIDEOMAE_PREDICTIONS_PATH,
    index=False,
)

display(
    videomae_predictions_df[
        [
            "video_id",
            "reference_class",
            "predicted_class",
            "confidence",
            "is_abnormal",
            "correct_prediction",
        ]
    ]
)

print(
    "Prédictions correctes :",
    int(videomae_predictions_df["correct_prediction"].sum()),
    "/ 12",
)
print("Temps total :", round(total_inference_time, 3), "s")
print("Temps moyen par vidéo :", round(mean_time_per_video, 3), "s")
print("CSV :", VIDEOMAE_PREDICTIONS_PATH)

## 6 — Détection contextuelle avec YOLOE-26s


In [ ]:
# ==============================================================================
# DÉTECTION CONTEXTUELLE OPEN-VOCABULARY AVEC YOLOE-26s
# ==============================================================================

import gc
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLOE


# ------------------------------------------------------------------------------
# 1. Configuration issue du benchmark
# ------------------------------------------------------------------------------

YOLOE_MODEL_NAME = "YOLOE-26s"
YOLOE_MODEL_ID = "yoloe-26s-seg.pt"
YOLOE_IMAGE_SIZE = 640
YOLOE_NMS_IOU = 0.50
YOLOE_MAX_DETECTIONS = 500

# Ces 43 classes sont exactement celles représentées dans les annotations
# du benchmark. La classe child est absente car l'enfant est déjà traité
# par le détecteur spécialisé YOLO26m.
YOLOE_CONTEXT_CLASSES = [
    "bed",
    "bench",
    "bookshelf",
    "bottle",
    "box",
    "cabinet",
    "cable",
    "chair",
    "cup",
    "cupboard",
    "curtain",
    "curtain_rod",
    "door",
    "dresser",
    "electrical_plug",
    "fan",
    "food_piece",
    "keys",
    "kitchen_utensil",
    "knife",
    "knife_block",
    "lamp",
    "marker",
    "medicine",
    "metal_bars",
    "oven",
    "pen",
    "railing",
    "refrigerator",
    "remote_control",
    "rug",
    "scissors",
    "shelf",
    "shoe",
    "sofa",
    "stairs",
    "table",
    "teddy_bear",
    "television",
    "television_stand",
    "toy",
    "toy_car",
    "wall",
]

YOLOE_TEXT_PROMPTS = [
    class_name.replace("_", " ")
    for class_name in YOLOE_CONTEXT_CLASSES
]

PROMPT_TO_CLASS = dict(
    zip(YOLOE_TEXT_PROMPTS, YOLOE_CONTEXT_CLASSES)
)

YOLOE_OBJECTS_DIR = PROJECT_DIR / "yoloe_26s_objects"
YOLOE_OBJECTS_DIR.mkdir(parents=True, exist_ok=True)

YOLOE_INFERENCE_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_inference_detections.csv"
)

YOLOE_LATENCY_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_frame_latency.csv"
)

assert OBJECT_CONFIDENCE_THRESHOLD == 0.14
assert len(YOLOE_CONTEXT_CLASSES) == 43
assert len(YOLOE_CONTEXT_CLASSES) == len(set(YOLOE_CONTEXT_CLASSES))
assert len(extracted_frames_df) == 192


# ------------------------------------------------------------------------------
# 2. NMS identique au protocole du benchmark
# ------------------------------------------------------------------------------

def box_iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    intersection_x1 = max(ax1, bx1)
    intersection_y1 = max(ay1, by1)
    intersection_x2 = min(ax2, bx2)
    intersection_y2 = min(ay2, by2)

    intersection = (
        max(0.0, intersection_x2 - intersection_x1)
        * max(0.0, intersection_y2 - intersection_y1)
    )

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def class_aware_nms(records, iou_threshold=YOLOE_NMS_IOU):
    kept_records = []
    records_by_class = defaultdict(list)

    for record in records:
        records_by_class[
            record["detected_class_name"]
        ].append(record)

    for class_records in records_by_class.values():
        remaining = sorted(
            class_records,
            key=lambda item: item["confidence"],
            reverse=True,
        )

        while remaining:
            best_record = remaining.pop(0)
            kept_records.append(best_record)

            best_box = [
                best_record[key]
                for key in (
                    "bbox_x1",
                    "bbox_y1",
                    "bbox_x2",
                    "bbox_y2",
                )
            ]

            remaining = [
                candidate
                for candidate in remaining
                if box_iou_xyxy(
                    best_box,
                    [
                        candidate[key]
                        for key in (
                            "bbox_x1",
                            "bbox_y1",
                            "bbox_x2",
                            "bbox_y2",
                        )
                    ],
                )
                < iou_threshold
            ]

    return kept_records


# ------------------------------------------------------------------------------
# 3. Chargement et configuration du modèle
# ------------------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()

object_model = YOLOE(YOLOE_MODEL_ID)
object_model.set_classes(YOLOE_TEXT_PROMPTS)

YOLOE_DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("DÉTECTEUR CONTEXTUEL")
print("=" * 80)
print("Modèle :", YOLOE_MODEL_NAME)
print("Checkpoint :", YOLOE_MODEL_ID)
print("Classes recherchées :", len(YOLOE_CONTEXT_CLASSES))
print(
    "Seuil de confiance calibré :",
    OBJECT_CONFIDENCE_THRESHOLD,
)
print("NMS IoU :", YOLOE_NMS_IOU)
print("Périphérique :", YOLOE_DEVICE)


# ------------------------------------------------------------------------------
# 4. Préchauffage GPU non inclus dans les temps mesurés
# ------------------------------------------------------------------------------

ordered_object_frames_df = (
    extracted_frames_df
    .sort_values(
        ["class_name", "video_id", "sample_index"]
    )
    .reset_index(drop=True)
)

warmup_frame_path = Path(
    ordered_object_frames_df.iloc[0]["frame_path"]
)

assert warmup_frame_path.exists()

object_model.predict(
    source=str(warmup_frame_path),
    conf=OBJECT_CONFIDENCE_THRESHOLD,
    iou=0.99,
    agnostic_nms=False,
    imgsz=YOLOE_IMAGE_SIZE,
    max_det=YOLOE_MAX_DETECTIONS,
    device=YOLOE_DEVICE,
    half=torch.cuda.is_available(),
    verbose=False,
)


# ------------------------------------------------------------------------------
# 5. Inférence sur les 192 frames
# ------------------------------------------------------------------------------

yoloe_detection_records = []
yoloe_latency_records = []

for frame_row in tqdm(
    ordered_object_frames_df.itertuples(index=False),
    total=len(ordered_object_frames_df),
    desc="Détection YOLOE-26s",
):
    frame_path = Path(frame_row.frame_path)

    assert frame_path.exists(), (
        f"Frame absente : {frame_path}"
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    result = object_model.predict(
        source=str(frame_path),
        conf=OBJECT_CONFIDENCE_THRESHOLD,
        iou=0.99,
        agnostic_nms=False,
        imgsz=YOLOE_IMAGE_SIZE,
        max_det=YOLOE_MAX_DETECTIONS,
        device=YOLOE_DEVICE,
        half=torch.cuda.is_available(),
        verbose=False,
    )[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed_seconds = time.perf_counter() - start_time

    yoloe_latency_records.append(
        {
            "video_id": str(frame_row.video_id),
            "sample_index": int(frame_row.sample_index),
            "frame_index": int(frame_row.frame_index),
            "frame_path": str(frame_path),
            "latency_seconds": float(elapsed_seconds),
        }
    )

    frame_records = []

    if result.boxes is not None and len(result.boxes) > 0:
        boxes = (
            result.boxes.xyxy
            .detach()
            .cpu()
            .numpy()
        )

        confidences = (
            result.boxes.conf
            .detach()
            .cpu()
            .numpy()
        )

        class_indices = (
            result.boxes.cls
            .detach()
            .cpu()
            .numpy()
            .astype(int)
        )

        for box, confidence, class_index in zip(
            boxes,
            confidences,
            class_indices,
        ):
            if not 0 <= class_index < len(
                YOLOE_CONTEXT_CLASSES
            ):
                continue

            class_name = YOLOE_CONTEXT_CLASSES[
                class_index
            ]

            x1, y1, x2, y2 = map(float, box)

            # Limitation aux dimensions de la frame.
            x1 = max(
                0.0,
                min(float(frame_row.width), x1),
            )
            y1 = max(
                0.0,
                min(float(frame_row.height), y1),
            )
            x2 = max(
                0.0,
                min(float(frame_row.width), x2),
            )
            y2 = max(
                0.0,
                min(float(frame_row.height), y2),
            )

            if x2 <= x1 or y2 <= y1:
                continue

            frame_records.append(
                {
                    "video_id": str(frame_row.video_id),
                    "true_activity_class": str(
                        frame_row.class_name
                    ),
                    "sample_index": int(
                        frame_row.sample_index
                    ),
                    "frame_index": int(
                        frame_row.frame_index
                    ),
                    "timestamp_seconds": float(
                        frame_row.timestamp_seconds
                    ),
                    "frame_path": str(frame_path),
                    "detected_class_name": class_name,
                    "confidence": float(confidence),
                    "bbox_x1": x1,
                    "bbox_y1": y1,
                    "bbox_x2": x2,
                    "bbox_y2": y2,
                    "detector_model": YOLOE_MODEL_NAME,
                }
            )

    # Suppression des détections redondantes d'une même classe.
    frame_records = class_aware_nms(
        frame_records,
        iou_threshold=YOLOE_NMS_IOU,
    )

    for detection_index, record in enumerate(
        frame_records
    ):
        record["detection_index"] = int(
            detection_index
        )
        yoloe_detection_records.append(record)


# ------------------------------------------------------------------------------
# 6. Création et sauvegarde des résultats
# ------------------------------------------------------------------------------

YOLOE_COLUMNS = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "timestamp_seconds",
    "frame_path",
    "detection_index",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
    "detector_model",
]

yoloe_inference_df = pd.DataFrame(
    yoloe_detection_records,
    columns=YOLOE_COLUMNS,
)

yoloe_latency_df = pd.DataFrame(
    yoloe_latency_records
)

yoloe_inference_df.to_csv(
    YOLOE_INFERENCE_PATH,
    index=False,
)

yoloe_latency_df.to_csv(
    YOLOE_LATENCY_PATH,
    index=False,
)


# ------------------------------------------------------------------------------
# 7. Vérifications minimales
# ------------------------------------------------------------------------------

assert len(yoloe_latency_df) == 192
assert not yoloe_inference_df.empty

assert yoloe_inference_df["confidence"].ge(
    OBJECT_CONFIDENCE_THRESHOLD
).all()

assert set(
    yoloe_inference_df["detected_class_name"]
).issubset(
    set(YOLOE_CONTEXT_CLASSES)
)

assert (
    yoloe_inference_df["bbox_x2"]
    > yoloe_inference_df["bbox_x1"]
).all()

assert (
    yoloe_inference_df["bbox_y2"]
    > yoloe_inference_df["bbox_y1"]
).all()


# ------------------------------------------------------------------------------
# 8. Résumé
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("INFÉRENCE YOLOE-26s TERMINÉE")
print("=" * 80)

print(
    "Frames traitées :",
    len(yoloe_latency_df),
)

print(
    "Détections après NMS :",
    len(yoloe_inference_df),
)

print(
    "Frames avec au moins un objet :",
    yoloe_inference_df["frame_path"].nunique(),
)

print(
    "Classes différentes détectées :",
    yoloe_inference_df[
        "detected_class_name"
    ].nunique(),
)

print(
    "Latence moyenne par frame :",
    f"{1000 * yoloe_latency_df['latency_seconds'].mean():.2f} ms",
)

print(
    "CSV des détections :",
    YOLOE_INFERENCE_PATH,
)

print(
    "CSV des latences :",
    YOLOE_LATENCY_PATH,
)


# ------------------------------------------------------------------------------
# 9. Libération de la mémoire GPU
# ------------------------------------------------------------------------------

del object_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7 — Contrôle qualité des détections YOLOE-26s


In [ ]:
# ==============================================================================
# CONTRÔLE QUALITÉ ET PRÉPARATION DES DÉTECTIONS YOLOE-26s
# ==============================================================================

from IPython.display import display


if "yoloe_inference_df" not in globals():
    yoloe_inference_df = pd.read_csv(YOLOE_INFERENCE_PATH)

required_object_columns = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "frame_path",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]

missing_object_columns = [
    column
    for column in required_object_columns
    if column not in yoloe_inference_df.columns
]
assert not missing_object_columns, (
    f"Colonnes YOLOE manquantes : {missing_object_columns}"
)

object_detections_df = yoloe_inference_df.copy()

numeric_columns = [
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]
for column in numeric_columns:
    object_detections_df[column] = pd.to_numeric(
        object_detections_df[column],
        errors="coerce",
    )

# Le nettoyage retire uniquement les données techniquement invalides. Aucun
# filtrage temporel arbitraire n'est appliqué : un objet dangereux visible sur
# une seule frame doit rester disponible pour le module d'évaluation du risque.
object_detections_df = (
    object_detections_df
    .dropna(subset=required_object_columns)
    .loc[
        lambda frame: (
            frame["confidence"].ge(OBJECT_CONFIDENCE_THRESHOLD)
            & frame["confidence"].le(1.0)
            & frame["bbox_x2"].gt(frame["bbox_x1"])
            & frame["bbox_y2"].gt(frame["bbox_y1"])
            & frame["detected_class_name"].isin(YOLOE_CONTEXT_CLASSES)
        )
    ]
    .drop_duplicates(
        subset=[
            "video_id",
            "sample_index",
            "detected_class_name",
            "bbox_x1",
            "bbox_y1",
            "bbox_x2",
            "bbox_y2",
        ]
    )
    .sort_values(
        [
            "video_id",
            "sample_index",
            "confidence",
        ],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

object_detections_df["bbox_width"] = (
    object_detections_df["bbox_x2"]
    - object_detections_df["bbox_x1"]
)
object_detections_df["bbox_height"] = (
    object_detections_df["bbox_y2"]
    - object_detections_df["bbox_y1"]
)
object_detections_df["detector_model"] = YOLOE_MODEL_NAME

# Résumé temporel : les absences ponctuelles dans certaines frames ne font pas
# disparaître l'objet du résumé vidéo. Le nombre de frames de présence est
# conservé pour le JSON final.
yoloe_temporal_summary_df = (
    object_detections_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
            "detected_class_name",
        ],
        as_index=False,
    )
    .agg(
        frames_present=("sample_index", "nunique"),
        detections_count=("detected_class_name", "size"),
        confidence_mean=("confidence", "mean"),
        confidence_max=("confidence", "max"),
    )
)
yoloe_temporal_summary_df["presence_ratio"] = (
    yoloe_temporal_summary_df["frames_present"]
    / NUM_CONTEXT_FRAMES
)

YOLOE_FINAL_DETECTIONS_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_final_detections.csv"
)
YOLOE_TEMPORAL_SUMMARY_PATH = (
    YOLOE_OBJECTS_DIR / "yoloe_26s_temporal_summary.csv"
)

object_detections_df.to_csv(
    YOLOE_FINAL_DETECTIONS_PATH,
    index=False,
)
yoloe_temporal_summary_df.to_csv(
    YOLOE_TEMPORAL_SUMMARY_PATH,
    index=False,
)

# Alias utilisé par la cellule d'export final.
FINAL_DETECTIONS_PATH = YOLOE_FINAL_DETECTIONS_PATH

assert not object_detections_df.empty
assert object_detections_df["confidence"].between(
    OBJECT_CONFIDENCE_THRESHOLD,
    1.0,
).all()
assert (object_detections_df["bbox_width"] > 0).all()
assert (object_detections_df["bbox_height"] > 0).all()
assert set(object_detections_df["detected_class_name"]).issubset(
    set(YOLOE_CONTEXT_CLASSES)
)

print("=" * 80)
print("CONTRÔLE QUALITÉ YOLOE-26s TERMINÉ")
print("=" * 80)
print("Détections retenues :", len(object_detections_df))
print(
    "Frames contenant au moins un objet :",
    object_detections_df["frame_path"].nunique(),
    "/ 192",
)
print(
    "Classes détectées :",
    object_detections_df["detected_class_name"].nunique(),
)
print("Détections finales :", YOLOE_FINAL_DETECTIONS_PATH)
print("Résumé temporel :", YOLOE_TEMPORAL_SUMMARY_PATH)

display(
    object_detections_df[
        "detected_class_name"
    ].value_counts().head(20)
)


## 8 — Fusion enfant–objets et calcul des distances


In [ ]:
# ==============================================================================
# CELLULE 7 — FUSION DESCRIPTIVE ENFANT–OBJETS
# Géométrie continue, sans décision near/far
# ==============================================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ==============================================================================
# 1. DOSSIERS ET FICHIERS DE SORTIE
# ==============================================================================

GEOMETRY_OUTPUT_DIR = (
    PROJECT_DIR
    / "child_object_geometry"
)

GEOMETRY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHILD_OBJECT_GEOMETRY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "child_object_geometry.csv"
)

OBJECT_TEMPORAL_SUMMARY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "object_temporal_summary.csv"
)

VIDEO_CONTEXT_SUMMARY_CSV_PATH = (
    GEOMETRY_OUTPUT_DIR
    / "video_context_summary.csv"
)


# ==============================================================================
# 2. VÉRIFICATIONS DES RÉSULTATS PRÉCÉDENTS
# ==============================================================================

assert (
    "child_frame_summary_fallback_df"
    in globals()
), (
    "Le tableau des détections enfant est absent."
)

assert (
    "object_detections_df"
    in globals()
), (
    "Le tableau des détections YOLOE-26s est absent."
)

assert (
    "extracted_frames_df"
    in globals()
), (
    "L'index des frames extraites est absent."
)

assert len(
    child_frame_summary_fallback_df
) == 192, (
    "192 résultats de détection enfant sont attendus."
)

assert not object_detections_df.empty, (
    "Le tableau des détections YOLOE-26s est vide."
)


# ==============================================================================
# 3. COLONNES UTILISÉES POUR LA FUSION
# ==============================================================================

merge_keys = [
    "video_id",
    "sample_index",
    "frame_index",
    "frame_path",
]

child_columns = (
    merge_keys
    + [
        "child_detected",
        "child_confidence",
        "child_bbox_x1",
        "child_bbox_y1",
        "child_bbox_x2",
        "child_bbox_y2",
        "child_detection_source",
    ]
)

missing_child_columns = [
    column
    for column in child_columns
    if column
    not in child_frame_summary_fallback_df.columns
]

assert not missing_child_columns, (
    "Colonnes enfant manquantes : "
    f"{missing_child_columns}"
)

required_object_columns = [
    "video_id",
    "true_activity_class",
    "sample_index",
    "frame_index",
    "timestamp_seconds",
    "frame_path",
    "detected_class_name",
    "confidence",
    "bbox_x1",
    "bbox_y1",
    "bbox_x2",
    "bbox_y2",
]

missing_object_columns = [
    column
    for column in required_object_columns
    if column
    not in object_detections_df.columns
]

assert not missing_object_columns, (
    "Colonnes objets manquantes : "
    f"{missing_object_columns}"
)


# ==============================================================================
# 4. RÉCUPÉRATION DES DIMENSIONS DES FRAMES
# ==============================================================================

required_frame_columns = [
    "frame_path",
    "width",
    "height",
]

missing_frame_columns = [
    column
    for column in required_frame_columns
    if column
    not in extracted_frames_df.columns
]

assert not missing_frame_columns, (
    "Colonnes de dimensions manquantes : "
    f"{missing_frame_columns}"
)

frame_dimensions_df = (
    extracted_frames_df[
        required_frame_columns
    ]
    .drop_duplicates(
        subset=["frame_path"]
    )
    .copy()
)

frame_dimensions_df[
    "width"
] = pd.to_numeric(
    frame_dimensions_df["width"],
    errors="coerce",
)

frame_dimensions_df[
    "height"
] = pd.to_numeric(
    frame_dimensions_df["height"],
    errors="coerce",
)

assert frame_dimensions_df[
    ["width", "height"]
].notna().all().all(), (
    "Certaines dimensions sont manquantes."
)

assert (
    frame_dimensions_df["width"] > 0
).all(), (
    "Certaines largeurs sont invalides."
)

assert (
    frame_dimensions_df["height"] > 0
).all(), (
    "Certaines hauteurs sont invalides."
)


# ==============================================================================
# 5. FUSION OBJETS + ENFANT + DIMENSIONS
# ==============================================================================

# Évite la création de width_x, width_y, height_x et height_y
# si cette cellule est exécutée plusieurs fois.

objects_for_merge_df = (
    object_detections_df
    .drop(
        columns=[
            "width",
            "height",
        ],
        errors="ignore",
    )
    .copy()
)

child_for_merge_df = (
    child_frame_summary_fallback_df[
        child_columns
    ]
    .copy()
)

child_object_context_df = (
    objects_for_merge_df
    .merge(
        child_for_merge_df,
        on=merge_keys,
        how="left",
        validate="many_to_one",
    )
    .merge(
        frame_dimensions_df,
        on="frame_path",
        how="left",
        validate="many_to_one",
    )
)

assert len(
    child_object_context_df
) == len(
    object_detections_df
), (
    "La fusion a modifié le nombre "
    "de détections d'objets."
)

assert child_object_context_df[
    ["width", "height"]
].notna().all().all(), (
    "Certaines détections n'ont pas "
    "de dimensions d'image."
)


# ==============================================================================
# 6. PRÉPARATION DES DIMENSIONS
# ==============================================================================

width = child_object_context_df[
    "width"
].astype(float)

height = child_object_context_df[
    "height"
].astype(float)

diagonal = np.hypot(
    width,
    height,
)

assert (
    diagonal > 0
).all(), (
    "Certaines diagonales d'image sont invalides."
)


# ==============================================================================
# 7. NORMALISATION DES BOUNDING BOXES DES OBJETS
# ==============================================================================

object_normalization = {
    "bbox_x1": width,
    "bbox_x2": width,
    "bbox_y1": height,
    "bbox_y2": height,
}

for coordinate, denominator in (
    object_normalization.items()
):

    normalized_column = (
        f"object_{coordinate}_normalized"
    )

    child_object_context_df[
        normalized_column
    ] = (
        child_object_context_df[
            coordinate
        ].astype(float)
        / denominator
    ).clip(
        0.0,
        1.0,
    )


# ==============================================================================
# 8. CALCUL DU CENTRE DES OBJETS
# ==============================================================================

child_object_context_df[
    "object_center_x"
] = (
    child_object_context_df["bbox_x1"]
    + child_object_context_df["bbox_x2"]
) / 2.0

child_object_context_df[
    "object_center_y"
] = (
    child_object_context_df["bbox_y1"]
    + child_object_context_df["bbox_y2"]
) / 2.0

child_object_context_df[
    "object_center_x_normalized"
] = (
    child_object_context_df[
        "object_center_x"
    ]
    / width
).clip(
    0.0,
    1.0,
)

child_object_context_df[
    "object_center_y_normalized"
] = (
    child_object_context_df[
        "object_center_y"
    ]
    / height
).clip(
    0.0,
    1.0,
)


# ==============================================================================
# 9. IDENTIFICATION DES FRAMES OÙ L'ENFANT EST DISPONIBLE
# ==============================================================================

child_bbox_columns = [
    "child_bbox_x1",
    "child_bbox_y1",
    "child_bbox_x2",
    "child_bbox_y2",
]

child_available = (
    child_object_context_df[
        "child_detected"
    ]
    .fillna(False)
    .astype(bool)
    &
    child_object_context_df[
        child_bbox_columns
    ]
    .notna()
    .all(axis=1)
)

child_object_context_df[
    "child_available_for_geometry"
] = child_available


# ==============================================================================
# 10. NORMALISATION DE LA BOUNDING BOX DE L'ENFANT
# ==============================================================================

child_normalization = {
    "child_bbox_x1": width,
    "child_bbox_x2": width,
    "child_bbox_y1": height,
    "child_bbox_y2": height,
}

for coordinate, denominator in (
    child_normalization.items()
):

    normalized_column = (
        f"{coordinate}_normalized"
    )

    child_object_context_df[
        normalized_column
    ] = np.nan

    child_object_context_df.loc[
        child_available,
        normalized_column,
    ] = (
        child_object_context_df.loc[
            child_available,
            coordinate,
        ].astype(float)
        / denominator[
            child_available
        ]
    ).clip(
        0.0,
        1.0,
    )


# ==============================================================================
# 11. CALCUL DU CENTRE DE L'ENFANT
# ==============================================================================

child_object_context_df[
    "child_center_x"
] = np.nan

child_object_context_df[
    "child_center_y"
] = np.nan

child_object_context_df.loc[
    child_available,
    "child_center_x",
] = (
    child_object_context_df.loc[
        child_available,
        "child_bbox_x1",
    ]
    +
    child_object_context_df.loc[
        child_available,
        "child_bbox_x2",
    ]
) / 2.0

child_object_context_df.loc[
    child_available,
    "child_center_y",
] = (
    child_object_context_df.loc[
        child_available,
        "child_bbox_y1",
    ]
    +
    child_object_context_df.loc[
        child_available,
        "child_bbox_y2",
    ]
) / 2.0

child_object_context_df[
    "child_center_x_normalized"
] = (
    child_object_context_df[
        "child_center_x"
    ]
    / width
)

child_object_context_df[
    "child_center_y_normalized"
] = (
    child_object_context_df[
        "child_center_y"
    ]
    / height
)


# ==============================================================================
# 12. DISTANCE ENTRE LES CENTRES
# ==============================================================================

child_object_context_df[
    "center_distance_pixels"
] = np.nan

child_object_context_df.loc[
    child_available,
    "center_distance_pixels",
] = np.hypot(
    (
        child_object_context_df.loc[
            child_available,
            "object_center_x",
        ]
        -
        child_object_context_df.loc[
            child_available,
            "child_center_x",
        ]
    ),
    (
        child_object_context_df.loc[
            child_available,
            "object_center_y",
        ]
        -
        child_object_context_df.loc[
            child_available,
            "child_center_y",
        ]
    ),
)

# Normalisation par la diagonale de l'image.

child_object_context_df[
    "center_distance_normalized"
] = (
    child_object_context_df[
        "center_distance_pixels"
    ]
    / diagonal
)


# ==============================================================================
# 13. DISTANCE MINIMALE ENTRE LES DEUX RECTANGLES
# ==============================================================================

# Cette distance vaut zéro lorsque les deux bounding boxes
# se touchent ou se chevauchent.

horizontal_gap = np.maximum.reduce(
    [
        (
            child_object_context_df[
                "child_bbox_x1"
            ].to_numpy()
            -
            child_object_context_df[
                "bbox_x2"
            ].to_numpy()
        ),
        (
            child_object_context_df[
                "bbox_x1"
            ].to_numpy()
            -
            child_object_context_df[
                "child_bbox_x2"
            ].to_numpy()
        ),
        np.zeros(
            len(child_object_context_df)
        ),
    ]
)

vertical_gap = np.maximum.reduce(
    [
        (
            child_object_context_df[
                "child_bbox_y1"
            ].to_numpy()
            -
            child_object_context_df[
                "bbox_y2"
            ].to_numpy()
        ),
        (
            child_object_context_df[
                "bbox_y1"
            ].to_numpy()
            -
            child_object_context_df[
                "child_bbox_y2"
            ].to_numpy()
        ),
        np.zeros(
            len(child_object_context_df)
        ),
    ]
)

child_object_context_df[
    "bbox_gap_pixels"
] = np.nan

child_object_context_df.loc[
    child_available,
    "bbox_gap_pixels",
] = np.hypot(
    horizontal_gap[
        child_available.to_numpy()
    ],
    vertical_gap[
        child_available.to_numpy()
    ],
)

child_object_context_df[
    "bbox_gap_normalized"
] = (
    child_object_context_df[
        "bbox_gap_pixels"
    ]
    / diagonal
)


# ==============================================================================
# 14. CRÉATION DES CLÉS DE FRAME
# ==============================================================================

child_object_context_df[
    "frame_key"
] = (
    child_object_context_df[
        "video_id"
    ].astype(str)
    + "__"
    + child_object_context_df[
        "sample_index"
    ].astype(str)
)

child_object_context_df[
    "frame_key_with_child"
] = (
    child_object_context_df[
        "frame_key"
    ]
    .where(child_available)
)


# ==============================================================================
# 15. TRI DES DÉTECTIONS
# ==============================================================================

child_object_context_df = (
    child_object_context_df
    .sort_values(
        [
            "video_id",
            "sample_index",
            "center_distance_normalized",
            "confidence",
        ],
        ascending=[
            True,
            True,
            True,
            False,
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ==============================================================================
# 16. RÉSUMÉ TEMPOREL PAR OBJET
# ==============================================================================

object_temporal_summary_df = (
    child_object_context_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
            "detected_class_name",
        ],
        as_index=False,
    )
    .agg(
        detection_count=(
            "detected_class_name",
            "size",
        ),
        frames_present=(
            "frame_key",
            "nunique",
        ),
        first_timestamp_seconds=(
            "timestamp_seconds",
            "min",
        ),
        last_timestamp_seconds=(
            "timestamp_seconds",
            "max",
        ),
        mean_confidence=(
            "confidence",
            "mean",
        ),
        maximum_confidence=(
            "confidence",
            "max",
        ),
        minimum_center_distance_normalized=(
            "center_distance_normalized",
            "min",
        ),
        mean_center_distance_normalized=(
            "center_distance_normalized",
            "mean",
        ),
        minimum_bbox_gap_normalized=(
            "bbox_gap_normalized",
            "min",
        ),
    )
)


# ==============================================================================
# 17. ESTIMATION DE LA DURÉE DE PRÉSENCE
# ==============================================================================

duration_lookup = (
    selected_videos_df[
        [
            "class_name",
            "duration_seconds",
        ]
    ]
    .rename(
        columns={
            "class_name": (
                "true_activity_class"
            ),
            "duration_seconds": (
                "video_duration_seconds"
            ),
        }
    )
)

object_temporal_summary_df = (
    object_temporal_summary_df
    .merge(
        duration_lookup,
        on="true_activity_class",
        how="left",
        validate="many_to_one",
    )
)

object_temporal_summary_df[
    "presence_ratio"
] = (
    object_temporal_summary_df[
        "frames_present"
    ]
    / NUM_CONTEXT_FRAMES
)

object_temporal_summary_df[
    "estimated_presence_seconds"
] = (
    object_temporal_summary_df[
        "presence_ratio"
    ]
    *
    object_temporal_summary_df[
        "video_duration_seconds"
    ]
)


# ==============================================================================
# 18. RÉSUMÉ DU CONTEXTE PAR VIDÉO
# ==============================================================================

video_context_summary_df = (
    child_object_context_df
    .groupby(
        [
            "video_id",
            "true_activity_class",
        ],
        as_index=False,
    )
    .agg(
        total_object_detections=(
            "detected_class_name",
            "size",
        ),
        object_classes_detected=(
            "detected_class_name",
            "nunique",
        ),
        frames_with_objects=(
            "frame_key",
            "nunique",
        ),
        frames_with_child_and_objects=(
            "frame_key_with_child",
            "nunique",
        ),
        minimum_center_distance_normalized=(
            "center_distance_normalized",
            "min",
        ),
    )
)


# ==============================================================================
# 19. SAUVEGARDE DES RÉSULTATS
# ==============================================================================

child_object_context_df.to_csv(
    CHILD_OBJECT_GEOMETRY_CSV_PATH,
    index=False,
)

object_temporal_summary_df.to_csv(
    OBJECT_TEMPORAL_SUMMARY_CSV_PATH,
    index=False,
)

video_context_summary_df.to_csv(
    VIDEO_CONTEXT_SUMMARY_CSV_PATH,
    index=False,
)


# ==============================================================================
# 20. TESTS DE COHÉRENCE
# ==============================================================================

assert (
    child_object_context_df[
        "center_distance_normalized"
    ]
    .dropna()
    .ge(0)
    .all()
), (
    "Certaines distances normalisées "
    "sont négatives."
)

assert (
    object_temporal_summary_df[
        "presence_ratio"
    ]
    .between(
        0,
        1,
    )
    .all()
), (
    "Certains ratios de présence "
    "sont invalides."
)

assert len(
    child_object_context_df
) == len(
    object_detections_df
)


# ==============================================================================
# 21. AFFICHAGE DES RÉSULTATS
# ==============================================================================

print("=" * 80)
print("FUSION DESCRIPTIVE TERMINÉE")
print("=" * 80)

print(
    "Détections d'objets :",
    len(child_object_context_df),
)

print(
    "Distances calculables :",
    int(child_available.sum()),
)

print(
    "Objets temporels résumés :",
    len(object_temporal_summary_df),
)

print(
    "Aucune décision near/far "
    "n'est prise dans le Module 3."
)

print(
    "Dimensions utilisées :",
    child_object_context_df[
        ["width", "height"]
    ]
    .drop_duplicates()
    .to_dict("records"),
)

print(
    "CSV géométrie :",
    CHILD_OBJECT_GEOMETRY_CSV_PATH,
)

print(
    "CSV temporel :",
    OBJECT_TEMPORAL_SUMMARY_CSV_PATH,
)

print(
    "CSV résumé vidéo :",
    VIDEO_CONTEXT_SUMMARY_CSV_PATH,
)

## 9 — JSON final

In [ ]:
# ==============================================================================
# CELLULE 8 — JSON FINAL PAR VIDÉO
# Sortie factuelle destinée au module LLM, sans décision de risque
# ==============================================================================

from pathlib import Path
import json
import math
import shutil
import time
import numpy as np
import pandas as pd


JSON_FULL_DIR = PROJECT_DIR / "json_outputs_final"
if JSON_FULL_DIR.exists():
    shutil.rmtree(JSON_FULL_DIR)
JSON_FULL_DIR.mkdir(parents=True, exist_ok=True)
COMBINED_JSON_FULL_PATH = JSON_FULL_DIR / "context_all_videos_final.json"


def to_float(value, digits=6):
    if value is None or pd.isna(value):
        return None
    return round(float(value), digits)


def to_int(value):
    if value is None or pd.isna(value):
        return None
    return int(value)


def bbox_values(row, prefix=""):
    keys = [f"{prefix}bbox_x1", f"{prefix}bbox_y1", f"{prefix}bbox_x2", f"{prefix}bbox_y2"]
    if any(pd.isna(row.get(key)) for key in keys):
        return None
    return [round(float(row[key]), 2) for key in keys]


def normalized_bbox_values(row, prefix=""):
    if prefix == "child_":
        keys = [
            "child_bbox_x1_normalized", "child_bbox_y1_normalized",
            "child_bbox_x2_normalized", "child_bbox_y2_normalized",
        ]
    else:
        keys = [
            "object_bbox_x1_normalized", "object_bbox_y1_normalized",
            "object_bbox_x2_normalized", "object_bbox_y2_normalized",
        ]
    if any(pd.isna(row.get(key)) for key in keys):
        return None
    return [round(float(row[key]), 6) for key in keys]


def object_detection_to_json(row, rank):
    return {
        "distance_rank": int(rank),
        "name": str(row["detected_class_name"]),
        "confidence": to_float(row["confidence"]),
        "bbox_pixels": bbox_values(row),
        "bbox_normalized": normalized_bbox_values(row),
        "center_normalized": [
            to_float(row["object_center_x_normalized"]),
            to_float(row["object_center_y_normalized"]),
        ],
        "distance_to_child": {
            "available": bool(row["child_available_for_geometry"]),
            "center_distance_pixels": to_float(row["center_distance_pixels"], 2),
            "center_distance_normalized": to_float(row["center_distance_normalized"]),
            "bbox_gap_pixels": to_float(row["bbox_gap_pixels"], 2),
            "bbox_gap_normalized": to_float(row["bbox_gap_normalized"]),
        },
    }


def temporal_object_to_json(row):
    return {
        "name": str(row["detected_class_name"]),
        "detection_count": int(row["detection_count"]),
        "frames_present": int(row["frames_present"]),
        "presence_ratio": to_float(row["presence_ratio"], 4),
        "estimated_presence_seconds": to_float(row["estimated_presence_seconds"], 3),
        "first_timestamp_seconds": to_float(row["first_timestamp_seconds"], 3),
        "last_timestamp_seconds": to_float(row["last_timestamp_seconds"], 3),
        "mean_confidence": to_float(row["mean_confidence"]),
        "maximum_confidence": to_float(row["maximum_confidence"]),
        "minimum_center_distance_normalized": to_float(
            row["minimum_center_distance_normalized"]
        ),
        "mean_center_distance_normalized": to_float(
            row["mean_center_distance_normalized"]
        ),
        "minimum_bbox_gap_normalized": to_float(
            row["minimum_bbox_gap_normalized"]
        ),
    }


assert len(videomae_predictions_df) == 12
assert len(child_frame_summary_fallback_df) == 192
assert len(child_object_context_df) == len(object_detections_df)

activity_lookup = videomae_predictions_df.set_index("video_id").to_dict("index")
yoloe_time_by_video = yoloe_latency_df.groupby("video_id")["latency_seconds"].sum().to_dict()

frame_time_per_video = float(FRAME_EXTRACTION_TOTAL_SECONDS) / 12.0
child_time_per_video = float(CHILD_INFERENCE_TOTAL_SECONDS) / 12.0

json_generation_start = time.perf_counter()
all_video_json_outputs = []

for video_row in selected_videos_df.itertuples(index=False):
    matching_frames = extracted_frames_df[
        extracted_frames_df["class_name"] == video_row.class_name
    ]
    assert matching_frames["video_id"].nunique() == 1
    video_id = str(matching_frames.iloc[0]["video_id"])
    assert video_id in activity_lookup

    activity = activity_lookup[video_id]
    child_frames = child_frame_summary_fallback_df[
        child_frame_summary_fallback_df["video_id"].astype(str) == video_id
    ].merge(
        extracted_frames_df[
            ["video_id", "sample_index", "frame_index", "frame_path", "width", "height"]
        ],
        on=["video_id", "sample_index", "frame_index", "frame_path"],
        how="left",
        validate="one_to_one",
    ).sort_values("sample_index")
    video_objects = child_object_context_df[
        child_object_context_df["video_id"].astype(str) == video_id
    ]
    temporal_objects = object_temporal_summary_df[
        object_temporal_summary_df["video_id"].astype(str) == video_id
    ].sort_values(
        ["minimum_center_distance_normalized", "frames_present"],
        ascending=[True, False],
        na_position="last",
    )

    frames_json = []
    for child_row in child_frames.itertuples(index=False):
        child_series = pd.Series(child_row._asdict())
        frame_objects = video_objects[
            video_objects["sample_index"] == int(child_row.sample_index)
        ].sort_values(
            ["center_distance_normalized", "confidence"],
            ascending=[True, False],
            na_position="last",
        )
        objects_json = [
            object_detection_to_json(object_row, rank)
            for rank, (_, object_row) in enumerate(frame_objects.iterrows(), start=1)
        ]

        child_detected = bool(child_row.child_detected)
        child_bbox_normalized = None
        child_center_normalized = None
        if child_detected:
            width_value = float(child_row.width)
            height_value = float(child_row.height)
            child_bbox_normalized = [
                to_float(child_row.child_bbox_x1 / width_value),
                to_float(child_row.child_bbox_y1 / height_value),
                to_float(child_row.child_bbox_x2 / width_value),
                to_float(child_row.child_bbox_y2 / height_value),
            ]
            child_center_normalized = [
                to_float((child_row.child_bbox_x1 + child_row.child_bbox_x2) / (2 * width_value)),
                to_float((child_row.child_bbox_y1 + child_row.child_bbox_y2) / (2 * height_value)),
            ]

        frames_json.append({
            "sample_index": int(child_row.sample_index),
            "frame_index": int(child_row.frame_index),
            "timestamp_seconds": to_float(child_row.timestamp_seconds, 3),
            "image_size": {"width": int(child_row.width), "height": int(child_row.height)},
            "child": {
                "detected": child_detected,
                "confidence": to_float(child_row.child_confidence),
                "detection_source": str(child_row.child_detection_source),
                "bbox_pixels": bbox_values(child_series, prefix="child_"),
                "bbox_normalized": child_bbox_normalized,
                "center_normalized": child_center_normalized,
            },
            "number_of_objects": len(objects_json),
            "objects_sorted_by_distance": objects_json,
        })

    video_json = {
        "video_id": video_id,
        "reference_activity_class": str(video_row.class_name),
        "video_duration_seconds": to_float(video_row.duration_seconds, 3),
        "activity": {
            "predicted_class": str(activity["predicted_class"]),
            "confidence": to_float(activity["confidence"]),
            "is_abnormal": bool(activity["is_abnormal"]),
            "source": "videomae_original_checkpoint",
        },
        "child_detection": {
            "frames_analyzed": len(child_frames),
            "frames_detected": int(child_frames["child_detected"].sum()),
            "detection_rate": to_float(child_frames["child_detected"].mean(), 4),
            "fallback_detections": int(
                child_frames["child_detection_source"].eq("fallback_threshold_0.10").sum()
            ),
        },
        "context": {
            "number_of_object_detections": len(video_objects),
            "number_of_object_classes": int(video_objects["detected_class_name"].nunique()),
            "temporal_object_summary": [
                temporal_object_to_json(row)
                for _, row in temporal_objects.iterrows()
            ],
            "geometry_policy": (
                "continuous normalized geometry only; no near/far or risk decision"
            ),
        },
        "frames": frames_json,
    }
    all_video_json_outputs.append(video_json)

JSON_GENERATION_TOTAL_SECONDS = time.perf_counter() - json_generation_start
json_time_per_video = JSON_GENERATION_TOTAL_SECONDS / 12.0

for video_json in all_video_json_outputs:
    video_id = video_json["video_id"]
    activity_seconds = float(activity_lookup[video_id]["mean_processing_time_per_video_seconds"])
    object_seconds = float(yoloe_time_by_video.get(video_id, 0.0))
    stage_times = {
        "frame_extraction": frame_time_per_video,
        "activity_inference_videomae": activity_seconds,
        "child_inference_yolo26m": child_time_per_video,
        "object_inference_yoloe_26s": object_seconds,
        "fusion_and_json_generation": json_time_per_video,
    }
    video_json["processing_time_seconds"] = {
        **{name: round(value, 4) for name, value in stage_times.items()},
        "total": round(sum(stage_times.values()), 4),
        "timing_scope": {
            "videomae": "subprocess startup, checkpoint loading and inference",
            "yolo26m": "GPU inference only, including fallback when used",
            "yoloe_26s": "GPU inference only after warm-up",
        },
        "excludes": [
            "dependency installation",
            "model download",
            "benchmark calibration",
            "visualization generation",
        ],
    }

for video_json in all_video_json_outputs:
    safe_video_id = video_json["video_id"].replace("/", "_").replace("\\", "_")
    with (JSON_FULL_DIR / f"context_{safe_video_id}_final.json").open(
        "w", encoding="utf-8"
    ) as json_file:
        json.dump(video_json, json_file, indent=2, ensure_ascii=False)

combined_json_full = {
    "module": "P3_context_extraction",
    "number_of_videos": len(all_video_json_outputs),
    "number_of_frames_per_video": NUM_CONTEXT_FRAMES,
    "total_object_detections": len(child_object_context_df),
    "decision_policy": (
        "descriptive output only; the following LLM module performs risk assessment"
    ),
    "activity_detector": {
        "model": "VideoMAE",
        "checkpoint": "original_training_checkpoint",
        "number_of_classes": 12,
        "output_policy": "raw prediction and confidence; no activity decision threshold",
    },
    "child_detector": {"model": "YOLO26m", "primary_threshold": 0.25, "fallback_threshold": 0.10},
    "object_detector": {
        "model": "YOLOE-26s",
        "checkpoint": YOLOE_MODEL_ID,
        "confidence_threshold": float(OBJECT_CONFIDENCE_THRESHOLD),
        "nms_iou_threshold": float(YOLOE_NMS_IOU),
        "number_of_prompt_classes": len(YOLOE_CONTEXT_CLASSES),
    },
    "videos": all_video_json_outputs,
}

with COMBINED_JSON_FULL_PATH.open("w", encoding="utf-8") as json_file:
    json.dump(combined_json_full, json_file, indent=2, ensure_ascii=False)

json_detection_count = sum(
    video["context"]["number_of_object_detections"]
    for video in all_video_json_outputs
)
assert json_detection_count == len(child_object_context_df)

print("=" * 80)
print("JSON FINAL CRÉÉ")
print("=" * 80)
print("JSON vidéo :", len(all_video_json_outputs))
print("Détections enregistrées :", json_detection_count)
print("Fichier combiné :", COMBINED_JSON_FULL_PATH)
print("Décision near/far : absente")


## 10 — Tests et export

In [ ]:
# ==============================================================================
# CELLULE 10 — TESTS UNITAIRES, D'INTÉGRATION ET SYSTÈME
# ==============================================================================

from pathlib import Path
import json
import math
import shutil
import pandas as pd


test_records = []


def validate(test_type, test_name, condition, details=""):
    passed = bool(condition)
    test_records.append({
        "type": test_type,
        "test": test_name,
        "passed": passed,
        "details": str(details),
    })
    assert passed, f"{test_type} — {test_name}: {details}"


# Tests unitaires
validate("unit", "to_float", to_float(1.234567, 4) == 1.2346)
validate("unit", "to_int", to_int(7.0) == 7)
validate("unit", "IoU identique", abs(box_iou_xyxy([0, 0, 10, 10], [0, 0, 10, 10]) - 1) < 1e-9)
validate("unit", "IoU disjointe", box_iou_xyxy([0, 0, 10, 10], [20, 20, 30, 30]) == 0)

# Tests des modules
validate("module", "12 vidéos sélectionnées", selected_videos_df["class_name"].nunique() == 12)
validate("module", "192 frames extraites", len(extracted_frames_df) == 192)
validate("module", "16 frames par vidéo", extracted_frames_df.groupby("video_id").size().eq(16).all())
validate("module", "192 résultats enfant", len(child_frame_summary_fallback_df) == 192)
validate("module", "confiances enfant valides", child_frame_summary_fallback_df["child_confidence"].dropna().between(0, 1).all())
validate("module", "12 prédictions VideoMAE", len(videomae_predictions_df) == 12)
validate("module", "confiances VideoMAE valides", videomae_predictions_df["confidence"].between(0, 1).all())
validate("module", "objets YOLOE disponibles", not object_detections_df.empty)
validate("module", "seuil YOLOE respecté", object_detections_df["confidence"].between(OBJECT_CONFIDENCE_THRESHOLD, 1).all())
validate("module", "classes YOLOE autorisées", set(object_detections_df["detected_class_name"]).issubset(set(YOLOE_CONTEXT_CLASSES)))
validate("module", "bounding boxes valides", ((object_detections_df["bbox_x2"] > object_detections_df["bbox_x1"]) & (object_detections_df["bbox_y2"] > object_detections_df["bbox_y1"])).all())
validate("module", "distances continues non négatives", child_object_context_df["center_distance_normalized"].dropna().ge(0).all())
validate("module", "coordonnées objets normalisées", child_object_context_df[["object_bbox_x1_normalized", "object_bbox_y1_normalized", "object_bbox_x2_normalized", "object_bbox_y2_normalized"]].apply(lambda column: column.between(0, 1).all()).all())
validate("module", "ratios de présence valides", object_temporal_summary_df["presence_ratio"].between(0, 1).all())
validate("module", "durées de présence valides", object_temporal_summary_df["estimated_presence_seconds"].ge(0).all())

# Tests d'intégration
video_ids_frames = set(extracted_frames_df["video_id"].astype(str))
video_ids_child = set(child_frame_summary_fallback_df["video_id"].astype(str))
video_ids_activity = set(videomae_predictions_df["video_id"].astype(str))
video_ids_json = {video["video_id"] for video in all_video_json_outputs}
validate("integration", "identifiants cohérents", video_ids_frames == video_ids_child == video_ids_activity == video_ids_json)
validate("integration", "12 JSON vidéo", len(all_video_json_outputs) == 12)
validate("integration", "16 frames par JSON", all(len(video["frames"]) == 16 for video in all_video_json_outputs))
validate("integration", "toutes les détections conservées", sum(video["context"]["number_of_object_detections"] for video in all_video_json_outputs) == len(child_object_context_df))
validate("integration", "résumé temporel présent", all("temporal_object_summary" in video["context"] for video in all_video_json_outputs))

# Tests système
serialized_json = json.dumps(combined_json_full, ensure_ascii=False)
validate("system", "aucune activité temporaire", "dataset_label_temporary" not in serialized_json)
validate("system", "trois modèles documentés", combined_json_full["activity_detector"]["model"] == "VideoMAE" and combined_json_full["child_detector"]["model"] == "YOLO26m" and combined_json_full["object_detector"]["model"] == "YOLOE-26s")
validate("system", "aucune décision near/far", '"near"' not in serialized_json and '"far"' not in serialized_json and "near_distance_threshold" not in serialized_json)
validate("system", "temps par étape complets", all(video["processing_time_seconds"]["total"] > 0 for video in all_video_json_outputs))
validate(
    "system",
    "un fichier JSON par vidéo",
    len([
        path for path in JSON_FULL_DIR.glob("context_*_final.json")
        if path.name != COMBINED_JSON_FULL_PATH.name
    ]) == 12,
)
validate("system", "JSON combiné lisible", COMBINED_JSON_FULL_PATH.exists())

tests_df = pd.DataFrame(test_records)
TEST_RESULTS_PATH = PROJECT_DIR / "pipeline_test_results.csv"
tests_df.to_csv(TEST_RESULTS_PATH, index=False)

FINAL_EXPORT_DIR = Path("/kaggle/working/P3_pipeline_final_results")
if FINAL_EXPORT_DIR.exists():
    shutil.rmtree(FINAL_EXPORT_DIR)
FINAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

files_to_export = [
    COMBINED_JSON_FULL_PATH,
    TEST_RESULTS_PATH,
    VIDEOMAE_PREDICTIONS_PATH,
    FALLBACK_FRAME_SUMMARY_PATH,
    YOLOE_FINAL_DETECTIONS_PATH,
    YOLOE_LATENCY_PATH,
    CHILD_OBJECT_GEOMETRY_CSV_PATH,
    OBJECT_TEMPORAL_SUMMARY_CSV_PATH,
    VIDEO_CONTEXT_SUMMARY_CSV_PATH,
    SELECTED_VIDEOS_PATH,
    EXTRACTED_FRAMES_INDEX_PATH,
]
for file_path in files_to_export:
    file_path = Path(file_path)
    if file_path.exists():
        shutil.copy2(file_path, FINAL_EXPORT_DIR / file_path.name)

shutil.copytree(JSON_FULL_DIR, FINAL_EXPORT_DIR / "json_per_video")
FINAL_ARCHIVE_PATH = Path("/kaggle/working/P3_pipeline_final_results.zip")
if FINAL_ARCHIVE_PATH.exists():
    FINAL_ARCHIVE_PATH.unlink()
shutil.make_archive(str(FINAL_ARCHIVE_PATH.with_suffix("")), "zip", FINAL_EXPORT_DIR)

print("=" * 80)
print("TESTS DU PIPELINE FINAL")
print("=" * 80)
display(tests_df)
print("Tests réussis :", int(tests_df["passed"].sum()), "/", len(tests_df))
print("JSON final :", COMBINED_JSON_FULL_PATH)
print("Archive :", FINAL_ARCHIVE_PATH)
